# 🫀 AI 的資料前處理基礎：動手做做看
**兩小時工作坊｜給零程式背景的醫學生**

> 今天不教你寫程式。程式都已經寫好了，你只需要 **按 ▶ 執行（或 Shift+Enter）**，並在標示 **👉 你的任務** 的地方動手。
> 目標是讓你親眼看到：**資料怎麼處理，會直接決定 AI 學到什麼、你能不能相信它。**

**今天的流程**


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">單元</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">內容</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">你會看到</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">0</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">準備環境</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">授權 Google Drive</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">1</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">資料如何才能「餵進去」</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">髒資料讓模型壞掉；你手動清理 vs 程式自動清理</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">2</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">資料怎麼分配</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">訓練／驗證／測試、交叉驗證、資料洩漏</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">3</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">前處理如何影響訓練</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">One-hot、標準化、擴增、資料不平衡</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">4</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">推論</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">新病人進來，前處理必須跟訓練時一模一樣</td></tr></table>


**資料來源**：UCI Heart Disease（Cleveland 資料集），Detrano et al., *Am J Cardiol* 1989；ISLR 整理版；CC BY 4.0。
本 notebook 裡的「錯誤」是老師**刻意製造**的，用於教學；模型僅供學習，**不可用於臨床**。

In [ ]:
#@title 投影片｜自我介紹
from IPython.display import HTML, display
display(HTML('<img src="https://raw.githubusercontent.com/Po-yuLin/MedicalAICourse_DataPreprocessing/main/slides/intro.png" style="width:100%;max-width:1100px;height:auto">'))

## 單元 0｜準備環境

In [ ]:
#@title 0-1　準備環境（會跳出 Google Drive 授權視窗，請按「允許」）
FONT_PX = 16       # 👈 表格與文字輸出的字體大小。建議：瀏覽器放大到 150%，這裡維持 16；表格太寬時會自動縮小配合視窗
import os, sys, io, json, time, base64, zlib, warnings, urllib.request
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

REPO = "Po-yuLin/MedicalAICourse_DataPreprocessing"      # 本課程的 GitHub repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/AI_preprocessing_class"
else:                                            # 不在 Colab 時（老師測試用）
    WORKDIR = os.path.join(os.path.expanduser("~"), "AI_preprocessing_class")
os.makedirs(WORKDIR, exist_ok=True)
print("✅ 你的工作資料夾：", WORKDIR)
print("   之後所有檔案都會存在這裡，打開 Google 雲端硬碟就看得到。")

In [ ]:
#@title 0-2　工具箱（檢查器、顯示工具、小型神經網路、前處理管線；不用展開）
import torch, torch.nn as nn
import matplotlib.pyplot as plt
torch.set_num_threads(2)
import re
# ===== 資料字典與規則（檢查器與自動清理共用）=====
NUM_COLS = ["age", "resting_bp", "chol", "max_hr", "oldpeak", "ca"]          # 連續數值／計數
BIN_COLS = ["fbs", "exang"]                                                  # 0/1
CODE_COLS = ["rest_ecg", "slope"]                                            # 用數字編碼的類別
CAT_COLS = {"sex": ["male", "female"],
            "chest_pain": ["typical", "nontypical", "nonanginal", "asymptomatic"],
            "thal": ["normal", "fixed", "reversible"],
            "disease": ["yes", "no"]}
RANGES = {"age": (18, 100), "resting_bp": (70, 250), "chol": (100, 600),
          "max_hr": (60, 220), "oldpeak": (0, 8), "ca": (0, 3),
          "fbs": (0, 1), "exang": (0, 1), "rest_ecg": (0, 2), "slope": (1, 3)}
ALL_NUMERIC = NUM_COLS + BIN_COLS + CODE_COLS
MISSING_TOKENS = {"", "?", "na", "n/a", "nan", "null", "none", "unknown", "-", "--"}


def _is_missing(x):
    return pd.isna(x) or str(x).strip().lower() in MISSING_TOKENS


def _num(v):
    try:
        return float(str(v).strip())
    except ValueError:
        return None


CODE_ALLOWED = {"fbs": {0, 1}, "exang": {0, 1}, "rest_ecg": {0, 1, 2}, "slope": {1, 2, 3}, "ca": {0, 1, 2, 3}}
ISSUE_ORDER = ["缺值", "數字欄位混入文字", "不可能的數值", "單位疑似錯誤", "類別寫法不一致", "重複列",
               "不該改的被改了", "這格有固定答案，但填的不對", "填的值不合理", "誤刪正常的病人", "多出不認識的病人"]


def _cell_kind(col, dirty_val, truth_val):
    """這一格原本是：clean（本來就對）/ fixed（有固定答案）/ fixed_unit（單位換算）/ imputable（缺值或不可能值，沒有唯一答案）"""
    tv = "" if pd.isna(truth_val) else str(truth_val).strip()
    if tv == "":
        return "imputable"
    dv = "" if pd.isna(dirty_val) else str(dirty_val)
    if col in ALL_NUMERIC:
        x = _num(dv)
        if x is not None and abs(x - float(tv)) < 1e-9:
            return "clean"
        m = re.search(r"-?\d+\.?\d*", dv)
        if x is None and m and not _is_missing(dv) and abs(float(m.group()) - float(tv)) < 1e-9:
            return "fixed"
        if col == "chol" and x is not None and 0 < x < 30 and abs(x * 38.67 - float(tv)) <= 2:
            return "fixed_unit"
        return "imputable"
    if dv == tv:
        return "clean"
    return "imputable" if _is_missing(dv) else "fixed"


def _accept(col, v, kind, truth_val, bands):
    """學生填的值可不可以接受？有固定答案的要一模一樣；沒有唯一答案的，只要『合理』。"""
    if kind in ("clean", "fixed", "fixed_unit"):
        if col in ALL_NUMERIC:
            x = _num(v)
            return x is not None and abs(x - float(truth_val)) <= (2 if kind == "fixed_unit" else 1e-9)
        return str(v) == str(truth_val)
    if col in ALL_NUMERIC:
        x = _num(v)
        if x is None:
            return False
        if col in CODE_ALLOWED:
            return x == int(x) and int(x) in CODE_ALLOWED[col]
        lo, hi = bands[col]
        return lo <= x <= hi
    return str(v) in CAT_COLS[col]


def find_issues(df, dirty=None, truth=None, bands=None):
    """找出表格裡所有問題。只給 df：只做「格式與範圍」檢查。
    再給 dirty（原始髒資料）、truth（答案）、bands（合理範圍）：還會檢查「不該改的被改了」「有固定答案但填錯」等。"""
    d = df.reset_index(drop=True)
    out = []
    def add(row, pid, col, typ, detail, expected=""):
        out.append(dict(row=row, patient_id=pid, column=col, type=typ, detail=detail, expected=expected))
    need = ["patient_id"] + ALL_NUMERIC + list(CAT_COLS)
    lost = [c for c in need if c not in d.columns]
    if lost:
        return None, lost
    for i, row in d.iterrows():
        pid = str(row["patient_id"]).strip()
        for col in ALL_NUMERIC:
            v = row[col]
            if _is_missing(v):
                add(i, pid, col, "缺值", "空白或 ?、NA、unknown…")
                continue
            x = _num(v)
            if x is None:
                add(i, pid, col, "數字欄位混入文字", f"「{v}」不是純數字")
                continue
            lo, hi = RANGES[col]
            if col == "chol" and 0 < x < 30:
                add(i, pid, col, "單位疑似錯誤", f"{x:g} 看起來是 mmol/L，要的是 mg/dL")
            elif not (lo <= x <= hi):
                add(i, pid, col, "不可能的數值", f"{x:g} 超出合理範圍 {lo}–{hi}")
        for col, allowed in CAT_COLS.items():
            v = row[col]
            if _is_missing(v):
                add(i, pid, col, "缺值", "空白或 NA…")
            elif str(v) not in allowed:
                add(i, pid, col, "類別寫法不一致", f"「{v}」不在 {allowed} 之中")
    for i in d.index[d.duplicated("patient_id", keep="first")]:
        add(i, str(d.at[i, "patient_id"]).strip(), "(整列)", "重複列", "同一位病人出現不只一次")

    deleted_info = None
    if truth is not None and dirty is not None:
        dirty_ix = dirty.assign(patient_id=dirty["patient_id"].str.strip()).drop_duplicates("patient_id").set_index("patient_id")
        truth_ix = truth.assign(patient_id=truth["patient_id"].str.strip()).set_index("patient_id")
        flagged = {(o["row"], o["column"]) for o in out}
        cols = ALL_NUMERIC + list(CAT_COLS)
        for i, row in d.iterrows():
            pid = str(row["patient_id"]).strip()
            if pid not in truth_ix.index:
                add(i, pid, "(整列)", "多出不認識的病人", "這個 patient_id 不在原始資料裡")
                continue
            for col in cols:
                if (i, col) in flagged:
                    continue
                tv, v = truth_ix.at[pid, col], row[col]
                kind = _cell_kind(col, dirty_ix.at[pid, col], tv)
                if _accept(col, v, kind, tv, bands):
                    continue
                if kind == "clean":
                    add(i, pid, col, "不該改的被改了", f"這格原本就是正常的，你把它改成了「{v}」", f"{tv}")
                elif kind in ("fixed", "fixed_unit"):
                    add(i, pid, col, "這格有固定答案，但填的不對", f"你填的是「{v}」", f"{tv}")
                else:
                    rng = f"{bands[col][0]:g}–{bands[col][1]:g}" if col in bands else "合法的類別"
                    add(i, pid, col, "填的值不合理", f"你填的是「{v}」，不在合理的範圍內", f"任何落在 {rng} 的值")
        present = set(d["patient_id"].astype(str).str.strip())
        gone = [p for p in truth_ix.index if p not in present]
        n_bad = 0
        for pid in gone:
            had_problem = any(_cell_kind(c, dirty_ix.at[pid, c], truth_ix.at[pid, c]) != "clean" for c in cols)
            if had_problem:
                n_bad += 1
            else:
                add(-1, pid, "(整列)", "誤刪正常的病人", "這位病人的資料原本完全正常，不該刪")
        deleted_info = (len(gone), n_bad)
    res = pd.DataFrame(out, columns=["row", "patient_id", "column", "type", "detail", "expected"])
    return res, deleted_info


def check_cleaned(df, hint=1, dirty=None, truth=None, bands=None):
    """檢查清理後的表格。hint=1 只報數量；2 加上欄位；3 再加上 patient_id；4 連答案都告訴你。"""
    res, info = find_issues(df, dirty, truth, bands)
    if res is None:
        print(f"❌ 找不到欄位 {info}。請不要更改標題列（第 1 列）的欄位名稱。")
        return False
    if truth is not None and info and info[0]:
        print(f"（你刪掉了 {info[0]} 位病人，其中 {info[1]} 位原本就有問題。）")
    if res.empty:
        print(f"✅ 全部通過！共 {len(df)} 列，沒有發現問題。")
        return True
    print(f"❌ 還有 {len(res)} 個問題：")
    for kind in [k for k in ISSUE_ORDER if k in set(res["type"])]:
        g = res[res["type"] == kind]
        line = f"  • {kind}：{len(g)} 個"
        if hint >= 2:
            line += f"　欄位：{', '.join(sorted(g['column'].unique()))}"
        print(line)
        if hint >= 3:
            for _, r in g.iterrows():
                extra = f"　→ 答案：{r['expected']}" if (hint >= 4 and r["expected"]) else ""
                print(f"      - {r['patient_id']} 的 {r['column']}：{r['detail']}{extra}")
    return False


def auto_clean(raw, fill=None):
    """用程式一次清完：同樣的規則，對 300 列跟 300 萬列都一樣快。
    fill：補缺值用的數字（中位數/眾數）。None＝由這批資料自己算；推論時要傳入「訓練時算好的」fill。"""
    d = raw.copy().astype(str)
    d = d.apply(lambda s: s.str.strip())
    d = d.replace({t: np.nan for t in ["", "?", "NA", "N/A", "na", "n/a", "null", "unknown", "-", "--"]})
    d = d.drop_duplicates(subset="patient_id", keep="first")           # 重複列
    for c in ALL_NUMERIC:                                              # 取出數字（去掉 mmHg、bpm…）
        d[c] = pd.to_numeric(d[c].str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce")
    d.loc[d["chol"] < 30, "chol"] *= 38.67                             # mmol/L → mg/dL
    for c, (lo, hi) in RANGES.items():                                 # 不可能的值 → 視為缺值
        d.loc[(d[c] < lo) | (d[c] > hi), c] = np.nan
    fix = {                                                            # 類別寫法統一
        "sex": {"m": "male", "male": "male", "f": "female", "female": "female"},
        "chest_pain": {"typical": "typical", "typical angina": "typical",
                       "nontypical": "nontypical", "non-typical": "nontypical", "atypical": "nontypical",
                       "nonanginal": "nonanginal", "non-anginal": "nonanginal", "non anginal": "nonanginal",
                       "asymptomatic": "asymptomatic"},
        "thal": {"normal": "normal", "fixed": "fixed", "fixed defect": "fixed",
                 "reversible": "reversible", "reversable": "reversible", "reversible defect": "reversible"},
        "disease": {"yes": "yes", "y": "yes", "1": "yes", "no": "no", "n": "no", "0": "no"},
    }
    cats = [c for c in CAT_COLS if c in d.columns]
    for c in cats:
        d[c] = d[c].str.lower().map(fix[c])
    if fill is None:                                                   # 補缺值：數字用中位數、類別用眾數
        fill = {c: d[c].median() for c in ALL_NUMERIC}
        fill.update({c: d[c].mode()[0] for c in cats})
    for c in ALL_NUMERIC + cats:
        d[c] = d[c].fillna(fill[c])
    d["chol"] = d["chol"].round(0)
    d[ALL_NUMERIC] = d[ALL_NUMERIC].astype(float)
    d = d.reset_index(drop=True)
    d.attrs["fill"] = fill
    return d

# ===== 投影用的顯示工具：大字表格、找碴標色、前後對照 =====
import html as _html
from IPython.display import display, HTML
if "FONT_PX" not in globals():
    FONT_PX = 16
if "TABLE_MAX_PX" not in globals():
    TABLE_MAX_PX = 1100        # 不在 Colab 時使用的預設寬度（Colab 內會自動偵測瀏覽器視窗寬度）
_builtin_print = print


def print(*args, sep=" ", end="\n", **kwargs):
    """把 print 換成『大字版』（字體大小由 FONT_PX 決定），方便投影。"""
    if kwargs.get("file") is not None:
        return _builtin_print(*args, sep=sep, end=end, **kwargs)
    text = sep.join(str(a) for a in args) or "\u00a0"
    display(HTML(f'<div style="font-size:{FONT_PX}px;line-height:1.55;white-space:pre-wrap;'
                 f'font-family:monospace;color:inherit">{_html.escape(text)}</div>'))


plt.rcParams.update({"font.size": FONT_PX * 0.75, "axes.titlesize": FONT_PX * 0.85, "axes.labelsize": FONT_PX * 0.8,
                     "legend.fontsize": FONT_PX * 0.7, "xtick.labelsize": FONT_PX * 0.7, "ytick.labelsize": FONT_PX * 0.7})


def _fmt(v):
    if v is None or (isinstance(v, (float, np.floating)) and pd.isna(v)):
        return "NaN"
    if isinstance(v, (float, np.floating)):
        return f"{v:g}"
    if isinstance(v, (int, np.integer)):
        return str(v)
    s = str(v)
    if s == "":
        return "⟨空白⟩"
    lead, trail = len(s) - len(s.lstrip()), len(s) - len(s.rstrip())
    return ("␣" * lead + s.strip() + "␣" * trail) if (lead or trail) else s


def _avail_px():
    """目前瀏覽器視窗可用寬度（CSS 像素）。瀏覽器放大到 150% 時，這個數字會變小，表格就會跟著縮小去配合。"""
    if globals().get("IN_COLAB"):
        try:
            from google.colab import output
            return max(500, int(output.eval_js("document.documentElement.clientWidth")) - 150)
        except Exception:
            pass
    return TABLE_MAX_PX


def _text_em(s):
    return sum(1.05 if ord(ch) > 0x2E80 else 0.68 for ch in s)


def show(df, styler=None, index=False):
    """顯示表格：字體最大 FONT_PX，但如果表格太寬，會自動縮小字體，讓整張表剛好放進視窗。
    ␣＝多餘的空格，⟨空白⟩＝空的格子。"""
    st = styler if styler is not None else df.style
    st = st.format(_fmt)
    if not index:
        try:
            st = st.hide(axis="index")
        except Exception:
            st = st.hide_index()
    # 估計整張表需要的寬度，決定字體大小
    cols = ([("" if df.index.name is None else str(df.index.name))] if index else []) + [str(c) for c in df.columns]
    widths = [_text_em(c) for c in cols]
    for j, c in enumerate(df.columns):
        k = j + (1 if index else 0)
        widths[k] = max([widths[k]] + [_text_em(_fmt(x)) for x in df[c].tolist()])
    if index:
        widths[0] = max([widths[0]] + [_text_em(str(x)) for x in df.index.tolist()])
    pad = 8                                                  # 每格左右內距（px，單邊）
    fit = 0.97 * (_avail_px() - 2 * pad * len(cols) - 8) / max(sum(widths), 1)
    px = max(10.0, min(float(FONT_PX), fit))
    st = st.set_table_styles([
        {"selector": "th", "props": [("font-size", f"{px:.1f}px"), ("padding", f"5px {pad}px"), ("text-align", "left"),
                                     ("background-color", "#e8eaf6"), ("color", "#000")]},
        {"selector": "td", "props": [("font-size", f"{px:.1f}px"), ("padding", f"5px {pad}px"),
                                     ("background-color", "#fff"), ("color", "#000")]}])
    display(HTML('<div style="max-width:100%;overflow-x:auto">' + st.to_html() + "</div>"))


def _legend(items):
    boxes = "".join(f'<span style="background:{c};color:#000;padding:4px 10px;margin-right:12px;border-radius:4px">{t}</span>'
                    for c, t in items)
    display(HTML(f'<div style="font-size:{FONT_PX * 0.85}px;margin:8px 0">{boxes}</div>'))


ISSUE_COLORS = {"缺值": "#ff8a80", "數字欄位混入文字": "#ff8a80", "不可能的數值": "#ffcc80", "單位疑似錯誤": "#ffcc80",
                "類別寫法不一致": "#fff59d", "重複列": "#bbdefb"}


def show_issues(df, rows=None):
    """把有問題的格子標上顏色。rows：只顯示這幾列（用 df 的列編號）。"""
    res, _ = find_issues(df)
    css = pd.DataFrame("", index=df.index, columns=df.columns)
    dup = df.duplicated("patient_id", keep=False)
    for i in df.index[dup]:
        css.loc[i, :] = "background-color:#bbdefb !important;color:#000"
    for r in res.itertuples():
        if r.column != "(整列)":
            css.loc[r.row, r.column] = f"background-color:{ISSUE_COLORS[r.type]} !important;color:#000;font-weight:bold"
    sub = df if rows is None else df.loc[rows]
    show(sub, styler=sub.style.apply(lambda _: css.loc[sub.index], axis=None))
    _legend([("#ff8a80", "紅：轉成數字時會變成 NaN（缺值、混入文字）"), ("#ffcc80", "橘：轉得成數字，但數值不可能或單位不對"),
             ("#fff59d", "黃：文字寫法不一致"), ("#bbdefb", "藍：重複的病人")])


def show_changes(before, after, title, cols=None, n=6, key="patient_id"):
    """列出這一步『改了哪些格子』：改之前 → 改之後。"""
    b = before.drop_duplicates(key).set_index(key)
    a = after.drop_duplicates(key).set_index(key)
    def same(x, y):
        if pd.isna(x) and pd.isna(y):
            return True
        if pd.isna(x) or pd.isna(y):
            return False
        fx, fy = _num(x), _num(y)
        return abs(fx - fy) < 1e-9 if (fx is not None and fy is not None) else str(x) == str(y)
    rows = []
    for pid in a.index.intersection(b.index):
        for c in (cols or list(a.columns)):
            if c in b.columns and not same(b.at[pid, c], a.at[pid, c]):
                rows.append({"patient_id": pid, "欄位": c, "改之前": b.at[pid, c], "改之後": a.at[pid, c]})
    print(f"▶ {title}　（共改了 {len(rows)} 格" + (f"，列出前 {n} 格）" if len(rows) > n else "）"))
    if rows:
        show(pd.DataFrame(rows).head(n))


# ===== 教學用圖表（SVG，用瀏覽器字型顯示，所以中文不會變方塊）=====
_FONT = "'Noto Sans TC','Microsoft JhengHei','PingFang TC','Heiti TC','Noto Sans CJK TC',sans-serif"
_C = {"blue": ("#dbeafe", "#2563eb"), "red": ("#fee2e2", "#dc2626"), "green": ("#dcfce7", "#16a34a"),
      "yellow": ("#fef9c3", "#ca8a04"), "gray": ("#f3f4f6", "#6b7280"), "purple": ("#ede9fe", "#7c3aed"),
      "orange": ("#ffedd5", "#ea580c"), "white": ("#ffffff", "#9ca3af")}


class _Svg:
    def __init__(self, w, h):
        self.w, self.h, self.p = w, h, [f'<rect width="{w}" height="{h}" fill="#ffffff"/>']

    def rect(self, x, y, w, h, c="gray", r=10, sw=2, dash=None, fill=None, stroke=None):
        f, s = _C[c]
        d = f' stroke-dasharray="{dash}"' if dash else ""
        self.p.append(f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="{r}" fill="{fill or f}" stroke="{stroke or s}" stroke-width="{sw}"{d}/>')

    def text(self, x, y, s, size=22, anchor="middle", weight="normal", fill="#111827", lh=1.35):
        lines = str(s).split("\n")
        tsp = "".join(f'<tspan x="{x}" dy="{0 if i == 0 else size * lh}">{_esc(t)}</tspan>' for i, t in enumerate(lines))
        self.p.append(f'<text x="{x}" y="{y}" font-size="{size}" text-anchor="{anchor}" font-weight="{weight}" fill="{fill}">{tsp}</text>')

    def line(self, x1, y1, x2, y2, color="#374151", sw=3, dash=None, arrow=False):
        d = f' stroke-dasharray="{dash}"' if dash else ""
        m = ' marker-end="url(#arr)"' if arrow else ""
        self.p.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{color}" stroke-width="{sw}"{d}{m}/>')

    def circle(self, x, y, r, c="gray", sw=2, fill=None):
        f, s = _C[c]
        self.p.append(f'<circle cx="{x}" cy="{y}" r="{r}" fill="{fill or f}" stroke="{s}" stroke-width="{sw}"/>')

    def html(self):
        return (f'<div style="max-width:100%;overflow-x:auto"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {self.w} {self.h}" '
                f'style="width:100%;max-width:{self.w}px;height:auto;font-family:{_FONT}">'
                '<defs><marker id="arr" markerUnits="userSpaceOnUse" markerWidth="14" markerHeight="14" refX="11" refY="7" orient="auto">'
                '<path d="M0,0 L14,7 L0,14 z" fill="#374151"/></marker></defs>' + "".join(self.p) + "</svg></div>")


def _esc(t):
    return str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _box(s, x, y, w, h, text, c="blue", size=21, weight="normal", r=10, sw=2, dash=None):
    s.rect(x, y, w, h, c, r, sw, dash)
    n = str(text).count("\n") + 1
    s.text(x + w / 2, y + h / 2 - (n - 1) * size * 0.675 + size * 0.35, text, size, weight=weight)


def fig_features_labels():
    s = _Svg(1000, 520)
    s.text(500, 42, "一位病人＝一列　一個欄位＝一個特徵　想預測的答案＝標籤", 27, weight="bold")
    cols = ["age", "sex", "chest_pain", "resting_bp", "chol", "max_hr", "disease"]
    rows = [["63", "male", "typical", "145", "233", "150", "no"], ["67", "male", "nonanginal", "160", "286", "108", "yes"],
            ["41", "female", "nontypical", "130", "204", "172", "no"]]
    x0, y0, cw, rh = 60, 175, 125, 52
    s.text(x0 + 3 * cw, 112, "特徵 features（X）", 26, weight="bold", fill="#1d4ed8")
    s.text(x0 + 3 * cw, 142, "我們「給模型看」的資料", 21, fill="#1d4ed8")
    s.line(x0, 160, x0 + 6 * cw - 6, 160, "#2563eb", 4)
    for j, c in enumerate(cols):
        lab = j == 6
        s.rect(x0 + j * cw, y0, cw - 6, rh, "red" if lab else "blue", 6)
        s.text(x0 + j * cw + (cw - 6) / 2, y0 + 34, c, 21, weight="bold")
        for i, r in enumerate(rows):
            s.rect(x0 + j * cw, y0 + (i + 1) * (rh + 4) + 4, cw - 6, rh, "red" if lab else "white", 6, 1.5)
            s.text(x0 + j * cw + (cw - 6) / 2, y0 + (i + 1) * (rh + 4) + 4 + 35, r[j], 21)
    lx = x0 + 6 * cw + (cw - 6) / 2
    s.line(lx, 470, lx, 440, "#dc2626", 4, arrow=True)
    s.text(975, 498, "標籤 label（y）：想預測的答案", 22, anchor="end", weight="bold", fill="#b91c1c")
    s.text(300, 465, "模型的工作：看 X → 猜 y\n（像實習醫師看過很多病歷，學著判斷診斷）", 22, fill="#374151")
    return s.html()


def fig_why_dirty():
    s = _Svg(1000, 500)
    s.text(500, 40, "為什麼電腦「拒絕」吃髒資料？", 28, weight="bold")
    _box(s, 40, 75, 250, 90, "原始資料\n（各系統匯出）", "gray", 22)
    _box(s, 375, 75, 250, 90, "轉成數字", "yellow", 24, "bold")
    _box(s, 710, 75, 250, 90, "模型\n（只吃數字）", "blue", 22)
    s.line(290, 120, 370, 120, arrow=True); s.line(625, 120, 705, 120, arrow=True)
    cards = [("red", "文字混在數字裡", '"54y"　"110 mmHg"\n→ 轉不成數字 → NaN'),
             ("red", "空白、?、NA、unknown", '"" "?" "NA"\n→ 全部變成 NaN'),
             ("orange", "轉得成，但是錯！", "膽固醇 5.46（mmol/L）\n被當成 5.46 mg/dL\n\"Male\" 被當成 0\n→ 不會報錯")]
    for i, (c, t, b) in enumerate(cards):
        x = 30 + i * 322
        s.rect(x, 210, 300, 165, c, 12)
        s.text(x + 150, 250, t, 22, weight="bold")
        s.text(x + 150, 292, b, 20)
        s.line(500, 165, x + 150, 208, "#9ca3af", 2, "6 5")
    s.rect(30, 395, 940, 90, "red", 12, 3)
    s.text(500, 432, "NaN 會「傳染」：1 ＋ NaN ＝ NaN　→　loss ＝ NaN　→　整個訓練壞掉", 24, weight="bold", fill="#991b1b")
    s.text(500, 466, "更可怕的是第三種：沒有任何警告，模型卻學到錯的東西", 21, fill="#7f1d1d")
    return s.html()


def fig_missing():
    s = _Svg(1000, 590)
    s.text(500, 40, "為什麼缺？缺值本身，可能就是資訊", 28, weight="bold")
    s.rect(30, 70, 455, 190, "green", 12)
    s.text(257, 108, "① 隨機缺", 25, weight="bold", fill="#166534")
    s.text(257, 150, "例：檢體溶血、系統漏存、單純忘了填\n\n缺的原因跟病情無關\n→ 刪除或填補通常比較安全", 20)
    s.rect(515, 70, 455, 190, "red", 12)
    s.text(742, 108, "② 因為「沒做這項檢查」而缺", 25, weight="bold", fill="#991b1b")
    s.text(742, 150, "例：胸痛病人沒驗心肌酵素（troponin）\n因為醫師覺得不像心肌梗塞\n\n「沒做」本身就帶有臨床資訊\n→ 隨便填補，會把這個訊號抹掉", 20)
    s.text(500, 302, "處理方式（只講概念）", 26, weight="bold")
    items = [("blue", "刪除", "整列或整欄拿掉", "簡單\n但損失資料；如果缺值\n有規律，會造成偏誤"),
             ("yellow", "填補", "中位數、眾數、預測值…", "保留資料\n但填進去的是「猜的」\n（要記得補值用的數字\n也只能從訓練資料算）"),
             ("purple", "標記", "多加一欄「是否缺值」", "讓模型自己學\n「缺」代表什麼\n（特別適合第②種）")]
    for i, (c, t, sub, body) in enumerate(items):
        x = 30 + i * 322
        s.rect(x, 322, 300, 250, c, 12)
        s.text(x + 150, 364, t, 26, weight="bold")
        s.text(x + 150, 396, sub, 19, fill="#374151")
        s.text(x + 150, 440, body, 19)
    return s.html()


def fig_tiny_net():
    s = _Svg(1000, 700)
    s.text(500, 38, "我們的超小神經網路：只有 4 個感知器", 28, weight="bold")
    ins = ["age", "male", "resting_bp", "chol", "fbs", "rest_ecg", "max_hr", "exang", "oldpeak", "slope", "ca"]
    ix, hx, ox = 150, 520, 830
    iy = [95 + i * 46 for i in range(len(ins))]
    hy = [210, 330, 450]
    oy = 330
    for y in iy:
        for h in hy:
            s.line(ix + 18, y, hx - 40, h, "#cbd5e1", 1.2)
    for h in hy:
        s.line(hx + 40, h, ox - 42, oy, "#94a3b8", 2.5)
    for n, y in zip(ins, iy):
        s.circle(ix, y, 16, "gray")
        s.text(ix - 28, y + 7, n, 19, anchor="end")
    for i, h in enumerate(hy):
        s.circle(hx, h, 40, "blue", 3)
        s.text(hx, h + 5, f"感知器 {i+1}", 17, weight="bold")
        s.text(hx, h + 24, "tanh", 15, fill="#1e40af")
    s.circle(ox, oy, 42, "red", 3)
    s.text(ox, oy + 2, "感知器 4", 17, weight="bold")
    s.text(ox, oy + 22, "sigmoid", 15, fill="#991b1b")
    s.line(ox + 44, oy, ox + 120, oy, "#dc2626", 3, arrow=True)
    s.text(ox + 46, oy - 52, "有病的機率\n（0 到 1）", 17, anchor="start", fill="#991b1b")
    s.text(ix, 612, "輸入：11 個特徵", 21, weight="bold")
    s.text(hx, 560, "隱藏層：3 個感知器", 21, weight="bold")
    s.text(ox, 560, "輸出層：1 個", 21, weight="bold")
    s.rect(150, 625, 700, 62, "yellow", 8)
    s.text(500, 652, "每個感知器：Σ（輸入 × 權重）＋ 偏差 → 激活函數", 19)
    s.text(500, 677, "參數：11×3＋3＋3＋1 ＝ 40 個（單元 4 的 22 個輸入則是 73 個）", 17, fill="#78350f")
    return s.html()


def _grid_cell(s, x, y, w, kind):
    s.rect(x, y, w, w, "red" if kind == "T" else "blue", 5, 2)


def fig_leak_patient():
    s = _Svg(1000, 520)
    s.text(500, 38, "洩漏 ①　病人層級：同一位病人的多次檢查，分到訓練與測試兩邊", 25, weight="bold")
    A = ["TXTTT", "XTTXT", "TTXTT", "TTTTX"]
    B = ["TTTTT", "XXXXX", "TTTTT", "TTTTT"]
    for k, (title, pat, x0, cc) in enumerate([("❌ 以「列」切分", A, 60, "red"), ("✅ 以「病人」切分", B, 560, "green")]):
        s.text(x0 + 190, 88, title, 25, weight="bold", fill="#991b1b" if k == 0 else "#166534")
        for i, row in enumerate(pat):
            s.text(x0 - 8, 148 + i * 66 + 8, f"P00{i+1}", 20, anchor="end")
            for j, ch in enumerate(row):
                _grid_cell(s, x0 + 10 + j * 74, 120 + i * 66, 58, "T" if ch == "X" else "B")
                s.text(x0 + 39 + j * 74, 156 + i * 66, f"#{j+1}", 17, fill="#374151")
    s.rect(60, 400, 14, 14, "blue", 3); s.text(84, 413, "訓練集", 19, anchor="start")
    s.rect(180, 400, 14, 14, "red", 3); s.text(204, 413, "測試集", 19, anchor="start")
    s.text(500, 462, "同一位病人的 5 次檢查長得非常像、答案也一樣。\n左邊：模型只要「認得這個人」就能答對測試題 → 成績虛高。", 21, fill="#374151")
    s.text(500, 508, "", 10)
    return s.html()


def fig_leak_preproc():
    s = _Svg(1000, 520)
    s.text(500, 38, "洩漏 ②　前處理：先對「全部」資料標準化，再切分", 26, weight="bold")
    lanes = [("❌ 洩漏", 80, "red", ["全部資料", "用全部資料\n算平均與標準差", "標準化", "切成\n訓練／測試"], 1),
             ("✅ 正確", 290, "green", ["全部資料", "先切成\n訓練／測試", "只用訓練集\n算平均與標準差", "套用到\n訓練與測試"], 2)]
    for title, y, c, names, hi in lanes:
        s.text(30, y - 8, title, 25, weight="bold", fill="#991b1b" if c == "red" else "#166534", anchor="start")
        for i, n in enumerate(names):
            x = 30 + i * 245
            _box(s, x, y + 10, 195, 100, n, c if i == hi else "gray", 20, "bold" if i == hi else "normal", 10, 3 if i == hi else 2)
            if i < 3:
                s.line(x + 197, y + 60, x + 243, y + 60, arrow=True)
    s.text(500, 235, "測試集的資訊已經「滲」進平均與標準差 → 模型偷看了考卷", 22, fill="#991b1b", weight="bold")
    s.text(500, 445, "任何「從資料算出來的東西」：平均、標準差、補缺值用的中位數、挑特徵…\n都只能用訓練集算，再「套用」到驗證、測試、新病人。", 22, fill="#166534")
    return s.html()


def fig_leak_feature():
    s = _Svg(1000, 560)
    s.text(500, 38, "洩漏 ③　特徵：不小心包含了「答案的線索」", 26, weight="bold")
    s.text(250, 85, "例一：影像上的醫院標記", 23, weight="bold")
    for k, (x, lab, c) in enumerate([(40, "A 醫院（重症較多）", "red"), (270, "B 醫院（門診）", "blue")]):
        s.rect(x, 105, 190, 190, c, 8)
        s.circle(x + 95, 205, 62, "white", 2)
        s.rect(x + 8, 112, 64, 28, "yellow", 4, 1.5)
        s.text(x + 40, 133, "ICU" if k == 0 else "OPD", 18, weight="bold")
        s.text(x + 95, 325, lab, 19)
    s.text(250, 372, "病人多的醫院，影像角落的字也特別多", 19, fill="#374151")
    s.line(250, 395, 250, 430, "#374151", 3, arrow=True)
    s.rect(40, 435, 420, 95, "orange", 10)
    s.text(250, 468, "模型學會「看角落的字」\n而不是「看肺部」→ 換家醫院就失靈", 21, weight="bold")
    s.text(745, 85, "例二：特徵發生在診斷「之後」", 23, weight="bold")
    s.line(560, 190, 950, 190, "#374151", 4, arrow=True)
    for x, t in [(600, "症狀出現"), (740, "診斷"), (880, "開始用藥")]:
        s.circle(x, 190, 9, "gray")
        s.text(x, 232, t, 20)
    s.line(670, 130, 670, 250, "#dc2626", 3, "8 6")
    s.text(670, 120, "預測的時間點", 19, fill="#dc2626", weight="bold")
    s.rect(780, 255, 190, 56, "red", 8)
    s.text(875, 291, "特徵：用藥紀錄", 20, weight="bold")
    s.rect(520, 340, 450, 190, "orange", 10)
    s.text(745, 380, "「正在使用硝化甘油」\n早就暗示了心絞痛的診斷\n\n特徵一定要是「預測當下」\n就拿得到的資訊", 21, weight="bold")
    return s.html()


def fig_imbalance():
    s = _Svg(1000, 540)
    s.text(500, 38, "類別不平衡：罕見疾病只占 1%", 28, weight="bold")
    for i in range(100):
        r, c = divmod(i, 10)
        sick = i == 37
        s.circle(70 + c * 34, 100 + r * 34, 12, "red" if sick else "gray", 2, fill="#dc2626" if sick else None)
    s.text(240, 470, "100 位病人裡，只有 1 位有病（紅點）", 20)
    s.rect(430, 100, 250, 270, "yellow", 12)
    s.text(555, 140, "模型只會一招：\n全部猜「沒病」", 22, weight="bold")
    s.text(555, 215, "accuracy＝99％", 24, weight="bold", fill="#166534")
    s.text(555, 268, "recall＝0％", 28, weight="bold", fill="#b91c1c")
    s.text(555, 310, "（有病的那一位，\n一個都沒抓到）", 19, fill="#7f1d1d")
    s.line(682, 235, 722, 235, arrow=True)
    s.rect(730, 80, 250, 400, "green", 12)
    s.text(855, 118, "怎麼辦？", 25, weight="bold")
    s.text(855, 165, "重抽樣\n過採樣：複製少數類別\n欠採樣：減少多數類別", 19)
    s.text(855, 275, "調整權重\nclass weight：漏掉\n少數類別，罰得更重", 19)
    s.text(855, 385, "選對指標\nrecall、precision、AUC\n別只看 accuracy", 19, weight="bold")
    return s.html()


def fig_scale():
    s = _Svg(1000, 520)
    s.text(500, 38, "標準化：把大家拉到「同一把尺」", 28, weight="bold")
    s.text(30, 95, "標準化之前（同一條數線）", 23, weight="bold", anchor="start")
    ax0, ax1 = 130, 940
    sx = lambda v: ax0 + (v - 0) / 130 * (ax1 - ax0)
    s.line(ax0, 235, ax1, 235, "#374151", 2)
    for v in (0, 40, 80, 120):
        s.line(sx(v), 230, sx(v), 241, "#374151", 2); s.text(sx(v), 262, str(v), 18)
    s.rect(sx(40), 120, sx(120) - sx(40), 36, "blue", 6)
    s.text(sx(80), 146, "體重　40–120 kg", 20, weight="bold")
    s.rect(sx(3.5), 170, max(sx(5.5) - sx(3.5), 8), 36, "red", 4)
    s.text(sx(5.5) + 12, 196, "血鉀　3.5–5.5 mmol/L（擠在一小點）", 20, anchor="start", weight="bold")
    s.text(500, 305, "體重的數字比血鉀大 20 倍以上 → 在模型眼中，「體重」的聲音大得多", 21, fill="#991b1b")
    s.text(30, 355, "標準化之後（減平均、除以標準差）", 23, weight="bold", anchor="start")
    bx0, bx1 = 130, 940
    tx = lambda v: bx0 + (v + 3) / 6 * (bx1 - bx0)
    s.line(bx0, 455, bx1, 455, "#374151", 2)
    for v in (-2, -1, 0, 1, 2):
        s.line(tx(v), 450, tx(v), 461, "#374151", 2); s.text(tx(v), 482, str(v), 18)
    s.rect(tx(-2), 385, tx(2) - tx(-2), 28, "blue", 6); s.text(tx(0), 407, "體重", 19, weight="bold")
    s.rect(tx(-2), 420, tx(2) - tx(-2), 28, "red", 6); s.text(tx(0), 442, "血鉀", 19, weight="bold")
    s.text(500, 510, "兩個變數都落在差不多的範圍 → 公平競爭", 20, fill="#166534")
    return s.html()


def fig_chain():
    s = _Svg(1000, 410)
    s.text(500, 36, "一個感知器：往前算（預測），再把 loss 一路「傳回去」（連鎖律）", 25, weight="bold")
    s.p.append('<g transform="translate(0,-60)">')
    s.circle(70, 150, 34, "blue"); s.text(70, 157, "x₁", 24)
    s.text(70, 205, "體重", 18)
    s.circle(70, 290, 34, "red"); s.text(70, 297, "x₂", 24)
    s.text(70, 345, "血鉀", 18)
    _box(s, 235, 190, 190, 70, "z＝w₁x₁＋w₂x₂＋b", "yellow", 18)
    s.line(105, 150, 235, 215, arrow=True); s.line(105, 290, 235, 240, arrow=True)
    s.text(172, 168, "×w₁", 19, fill="#1d4ed8"); s.text(172, 278, "×w₂", 19, fill="#b91c1c")
    _box(s, 470, 190, 130, 70, "ReLU\ny＝max(z, 0)", "purple", 18)
    s.line(425, 225, 470, 225, arrow=True)
    _box(s, 680, 190, 170, 70, "loss＝(y − t)²\nt＝正確答案", "orange", 18)
    s.line(600, 225, 680, 225, arrow=True)
    s.text(905, 232, "L", 30, weight="bold")
    s.line(850, 225, 885, 225, arrow=True)
    for x, t in [(765, "∂L/∂y ＝ 2(y−t)"), (535, "∂y/∂z ＝ 1（z>0）或 0"), (325, "∂z/∂wᵢ ＝ xᵢ")]:
        s.line(x + 40 if x == 765 else x + 60, 285, x - 40 if x != 765 else x - 30, 285, "#dc2626", 3, arrow=True)
        s.text(x, 320, t, 20, fill="#b91c1c", weight="bold")
    s.p.append("</g>")
    s.rect(120, 312, 760, 84, "yellow", 12)
    s.text(500, 345, "梯度 ∂L/∂wᵢ ＝ 2(y−t) × ∂y/∂z × xᵢ　　更新：wᵢ ← wᵢ − 學習率 × 梯度", 21, weight="bold")
    s.text(500, 380, "輸入 xᵢ 越大，梯度越大，這個權重被「推」得越多", 21, fill="#991b1b", weight="bold")
    return s.html()


def fig_splits():
    s = _Svg(1000, 600)
    s.text(500, 36, "不同的「輪流當驗證」方式", 28, weight="bold")
    def panel(x0, title, sub, rounds, n, groups=None):
        s.text(x0 + 150, 80, title, 23, weight="bold")
        s.text(x0 + 150, 108, sub, 17, fill="#374151")
        cw = 300 / n
        for r, val in enumerate(rounds):
            y = 135 + r * 52
            for j in range(n):
                isv = j in val
                s.rect(x0 + j * cw, y, cw - 3, 36, "red" if isv else "blue", 3, 1.5)
            if groups:
                for g in range(1, n // groups):
                    s.line(x0 + g * groups * cw - 1.5, y - 4, x0 + g * groups * cw - 1.5, y + 40, "#111827", 2)
        return 135 + len(rounds) * 52
    y1 = panel(30, "K 折（K＝5）", "切成 5 份，輪流 1 份當驗證", [{0, 1}, {2, 3}, {4, 5}, {6, 7}, {8, 9}], 10)
    s.text(180, y1 + 28, "訓練 5 次", 20, weight="bold")
    y2 = panel(350, "Leave-one-out", "每次只留「1 個樣本」驗證", [{0}, {1}, {2}, {3}, {4}], 10)
    s.text(500, y2 + 4, "⋮", 26); s.text(500, y2 + 40, "N 個樣本 → 訓練 N 次", 20, weight="bold")
    y3 = panel(670, "Leave-one-patient-out", "每次留「1 位病人」（的所有列）驗證", [{0, 1, 2}, {3, 4, 5}, {6, 7, 8}, {9, 10, 11}, {12, 13, 14}], 15, 3)
    s.text(820, y3 + 28, "N 位病人 → 訓練 N 次", 20, weight="bold")
    s.rect(30, 470, 940, 110, "yellow", 12)
    s.text(500, 503, "資料很少時：Leave-one-out 讓訓練資料用得最多，但要訓練 N 次，而且每次驗證只有 1 筆，成績不穩定。", 19)
    s.text(500, 535, "一位病人有多筆資料時：一定要用 Leave-one-patient-out（或 K 折的「以病人分組」版本），否則會洩漏。", 19, weight="bold")
    s.text(500, 565, "紅＝驗證，藍＝訓練；黑色粗線分隔不同病人。", 17, fill="#374151")
    return s.html()


def fig_privacy():
    s = _Svg(1000, 520)
    s.text(500, 38, "研究資料裡，藏著哪些「可以認出病人」的東西？", 27, weight="bold")
    cards = [("purple", "🧑 影像：臉部", ["3D 頭部 MRI／CT 可以重建出臉，", "再用人臉辨識比對到真人", "", "對策：去臉（defacing）、", "只提供去除顱骨的腦組織"]),
             ("blue", "🗂 DICOM 標頭", ["姓名、病歷號、生日", "檢查日期、醫院、醫師", "機器序號、各種 UID", "", "對策：刪除／取代／日期位移", "UID 重新產生；小心影像內嵌文字"]),
             ("yellow", "📝 病歷文字", ["姓名、身分證字號、電話", "地址、日期、病歷號", "「王大明的女兒」", "", "對策：規則＋自然語言處理，", "再人工抽查"])]
    for i, (c, t, body) in enumerate(cards):
        x = 25 + i * 325
        s.rect(x, 70, 305, 250, c, 14)
        s.text(x + 152, 112, t, 24, weight="bold")
        s.text(x + 152, 158, "\n".join(body), 19)
    s.rect(25, 345, 950, 150, "red", 14)
    s.text(500, 382, "去識別化 ≠ 絕對安全", 25, weight="bold", fill="#991b1b")
    s.text(500, 420, "還有「間接識別」：罕見疾病 ＋ 精確日期 ＋ 居住地區，湊在一起也可能認出人。\n所以研究要經過 IRB，資料要依法處理，不要自己帶走、不要放到公開的 AI 服務。", 20)
    return s.html()


FIGS = {k[4:]: v for k, v in globals().items() if k.startswith("fig_") and callable(v)}


def show_fig(name):
    display(HTML(FIGS[name]()))


def slide(name, base=None):
    """顯示 repo 裡 slides/ 資料夾的投影片圖片。"""
    bases = [base] if base else [f"https://raw.githubusercontent.com/{REPO}/v1.0/slides/", f"https://raw.githubusercontent.com/{REPO}/main/slides/"]
    url = bases[-1] + name
    display(HTML(f'<div style="max-width:100%"><img src="{url}" alt="（找不到投影片 {name}：請把圖片放進 repo 的 slides/ 資料夾）" '
                 f'style="width:100%;max-width:1100px;height:auto"></div>'))

# ===== 小型 MLP 與評估工具 =====
import torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

FEATURES_V1 = NUM_COLS + ["male"] + BIN_COLS + CODE_COLS      # 單元一、二使用：11 個全數字欄位
CATEGORIES = {"chest_pain": CAT_COLS["chest_pain"], "thal": CAT_COLS["thal"],
              "rest_ecg": [0.0, 1.0, 2.0], "slope": [1.0, 2.0, 3.0]}


def add_male(df):
    d = df.copy()
    d["male"] = (d["sex"] == "male").astype(float)
    return d


ROUGH = {"age": 100, "resting_bp": 200, "chol": 400, "max_hr": 200, "oldpeak": 6, "ca": 3, "rest_ecg": 2, "slope": 3}


def to_xy(df, features=FEATURES_V1, rough=True):
    """轉成模型吃的數字矩陣。rough=True：每個欄位除以一個固定的數字，縮到大約 0–1（單元 3 會解釋為什麼）。"""
    d = add_male(df)
    X = d[features].astype("float32").values
    if rough:
        for j, c in enumerate(features):
            if c in ROUGH:
                X[:, j] /= ROUGH[c]
    y = (d["disease"] == "yes").astype("float32").values
    return X, y


class MLP(nn.Module):
    """hidden=(3,)：輸入 → 3 個隱藏感知器（tanh，一點點非線性）→ 1 個輸出感知器。共 4 個感知器。"""
    def __init__(self, n_in, hidden=(3,)):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X, y, epochs=300, lr=0.003, seed=0, pos_weight=None, X_val=None, y_val=None, hidden=(3,)):
    """整批資料一起訓練。回傳 (model, history)。history 裡有每個 epoch 的 loss。"""
    torch.manual_seed(seed)
    Xt, yt = torch.tensor(np.asarray(X), dtype=torch.float32), torch.tensor(np.asarray(y), dtype=torch.float32)
    model = MLP(Xt.shape[1], hidden)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    pw = None if pos_weight is None else torch.tensor(float(pos_weight))
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
    hist = {"train": [], "val": []}
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        opt.step()
        hist["train"].append(loss.item())
        if X_val is not None:
            model.eval()
            with torch.no_grad():
                hist["val"].append(nn.BCEWithLogitsLoss()(model(torch.tensor(np.asarray(X_val), dtype=torch.float32)),
                                                          torch.tensor(np.asarray(y_val), dtype=torch.float32)).item())
    return model, hist


def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(np.asarray(X), dtype=torch.float32))).numpy()


def evaluate(model, X, y, threshold=0.5):
    p = predict_proba(model, X)
    pred = (p >= threshold).astype(int)
    out = {"acc": accuracy_score(y, pred), "recall": recall_score(y, pred, zero_division=0),
           "precision": precision_score(y, pred, zero_division=0)}
    out["auc"] = roc_auc_score(y, p) if len(set(y)) > 1 and np.isfinite(p).all() else float("nan")
    return out

# ===== 前處理管線（單元三、四共用）=====
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, train_test_split, GroupShuffleSplit
import matplotlib.pyplot as plt

PREP_NUM = NUM_COLS                       # 要標準化的連續數值
PREP_BIN = ["male"] + BIN_COLS            # 本來就是 0/1
PREP_CAT = ["chest_pain", "thal", "rest_ecg", "slope"]   # 類別


def _rough_scale(X):
    return np.asarray(X, dtype="float64") / np.array([ROUGH[c] for c in PREP_NUM])


def make_prep(encode="onehot", scale="standard"):
    """encode: 'none'（類別欄位只用數字碼，且不用文字欄位）/ 'int'（文字類別轉成整數）/ 'onehot'
    scale: 'none' 不縮放 / 'rough' 除以固定數字 / 'standard'（或 True）標準化"""
    if scale is True:
        scale = "standard"
    if scale is False:
        scale = "none"
    num_tf = {"none": "passthrough", "rough": FunctionTransformer(_rough_scale), "standard": StandardScaler()}[scale]
    steps = [("num", num_tf, PREP_NUM),
             ("bin", "passthrough", PREP_BIN)]
    if encode == "none":
        steps.append(("code", "passthrough", CODE_COLS))
    elif encode == "int":
        steps.append(("code", "passthrough", CODE_COLS))
        steps.append(("ord", OrdinalEncoder(categories=[CATEGORIES["chest_pain"], CATEGORIES["thal"]]),
                      ["chest_pain", "thal"]))
    elif encode == "onehot":
        steps.append(("onehot", OneHotEncoder(categories=[CATEGORIES[c] for c in PREP_CAT], sparse_output=False),
                      PREP_CAT))
    return ColumnTransformer(steps)


RESULTS = []          # 單元三所有實驗的成績，最後一起比較


def cv_eval(df, label, encode="onehot", scale="standard", augment=None, epochs=300, lr=0.003, repeats=3, record=True):
    """5 折交叉驗證 × repeats 次。前處理「只用訓練折」來 fit，再套用到驗證折。"""
    d = add_male(df).reset_index(drop=True)
    y = (d["disease"] == "yes").astype(float).values
    rows = []
    for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=repeats, random_state=0).split(d, y)):
        prep = make_prep(encode, scale)
        Xtr = prep.fit_transform(d.iloc[tr]).astype("float32")
        Xva = prep.transform(d.iloc[va]).astype("float32")
        ytr = y[tr]
        if augment is not None:
            Xtr, ytr = augment(Xtr, ytr, k)
        model, _ = train_mlp(Xtr, ytr, epochs=epochs, lr=lr, seed=k)
        rows.append(evaluate(model, Xva, y[va]))
    r = pd.DataFrame(rows)
    out = dict(label=label, acc=r["acc"].mean(), acc_sd=r["acc"].std(), auc=r["auc"].mean(), auc_sd=r["auc"].std())
    if record:
        RESULTS.append(out)
    print(f"{label:<30} accuracy {out['acc']:.3f} ± {out['acc_sd']:.3f}   AUC {out['auc']:.3f} ± {out['auc_sd']:.3f}")
    return out


def noise_augment(copies=3, sd=0.1):
    """表格資料的擴增：把訓練資料複製幾份，連續數值加上一點雜訊。只能用在訓練集！"""
    def f(X, y, seed):
        r = np.random.default_rng(seed)
        mask = np.r_[np.ones(len(PREP_NUM)), np.zeros(X.shape[1] - len(PREP_NUM))].astype("float32")
        Xs, ys = [X], [y]
        for _ in range(copies):
            Xs.append(X + r.normal(0, sd, X.shape).astype("float32") * mask)
            ys.append(y)
        return np.vstack(Xs), np.concatenate(ys)
    return f

print("✅ 工具箱就緒（PyTorch", torch.__version__ + "）")

In [ ]:
#@title 0-3　載入資料（先從 GitHub 讀；讀不到就改用內嵌的備援資料）
_EMBEDDED = {
    "heart_dirty_small.csv":
        "eNqtVstu2zAQvPcr/AELh1yKlHgqjKJAC+SFNpeeDMVmHCG2bFhum/x9hxSl6sGcWhiwBEGzj9nZoU7lpXL1ZV1tqdw5atwrbZ5dc1mfyqqmM+6qerd+POHpcU9Pj014tnabHR3K1/XzmdxrWe/ouN+eXPlCzf54crQp6fJc7mlbNa5s3Id7ITRlkp7codw7qo/15e1UbfCGVIJYZIR/kjnjKpcZSVzr4xkv4+LRloyigF19/3Fz/3B3s3r4+qlF64jO8ojGPar85c5N9QjEm2sQgiWZrA3R55aC9DIzEe7TymVBPEmOHJlokWXzdjhdjgeQtglwafIWLT2aA3aWWqH3mBqdg62q7jpXCgCEya3vOdG5MpRxKjlKYjYRXbTXCRZJtZ3n1UBKDm9L3XJmZlitySSbDozHqiNl4X/eti78yJO8LQ6HL7s4b1Si/9Y/j2JJJ8hjTflSxNFpH0csNSnEGHYBYvRQdT0c7OVL202+lZ0ObYzglrKCUpIVLbIId0ueaaZAYpVKzNCXjHml7tuOKDtATWgHTkWhG5Ect2UynEYbr9RuTXxWg5oVflO6pVAETSSDoHQlVGTcxhJGhEm/p8mRcxw5d7VnHqwgvOmyIopfS4b4RjEW4MtzZ3u8n1l646SUvo5kF2g/8mBC+WP8Yuue3OYSw7Dy8pnNUPjV41iHejwdwhbMROAJ66Q7Mw5WsRH2ZXCwnfkO/Y8QilNNWNgk5y1et7YzWWLhsXCuTpCLIfxn/VIff9eYR7QCo9Ob4PnWRYJDTZxBFLur7XWUlK/i9mo1nWiIgrUwvSxvB3GwQ9oM90IGTc0jAA6buFldfx6bMFjjrodO1KMGCtmbwETToDDvUpvoQEk5FiMP630EC8tFXCirRb9RM7yFGRdJN/FWLgduImMJgwYYUnnPF/xRkGUDFbTHZ48P6Rk2iymmdgkmImx/+raFzE3l3ysA1VolZ6BAYbuKVscTmOl2FXHQL4ImiDP+q0PGsauUlzFO3vTqsR9bZ+Em2FhMe/cN+upSQ2/mvQO0UKFW77YyKFYlZOP3O7PJbyavvK56jofReOgYh06pVsRjxNPdalZONfcjJIfgUh8P8BqGDwe/aM+j0PxHeqpe3TYWnuseO6pZZMOal0UgYTRoK3rWhg17nWditKVq1PAfqZkQew==",
    "heart_dirty_full.csv":
        "eNqtXMuOXrcN3vcp5gGO/4q6axUMigYNkJkEaTZeGRN77AziGRsZt03evqRE6ehCzaLuJg4S80iiyI/kR+r/fPfl4f7py5uHd8fdh/vj+f6P4+2v989f3ny+e3g6fsd/e3j68OaXz/hfP3083v/ynP/bm/u3H47Huz/e/Pr7cf/H3dOH49PHd5/v7347nj9++nx/vL07vvx69/F49/B8f/d8/5cflYLDGxT5eH98+fPzw1v8n2DdoY054NAHOHWoQ1/MYfDP9w9/3L87nj6RnD58KHJ3z38+fv7y6RF3/PYAj389ehI6QEX8CFzwcyh+++l3/OvHn/fPJG424hoFdSri+Cfu4eLx3zUe79/3vz8//IIS/Al7GP7E06cnPOvDE23eoGjetDogBvynwfVp809l/bJ7d1g43t9X6XZ0ksYP5+WDpm9cLG5ikvaH88fjIEj7Nr4sGyL9eYlZ8LYXDIfXddnrf76++fHnH26uf/7ub6hzlPexLOxV3rbHbeu6MJ85Hi7UDyyKM86WDXi6PIUf6HZ+mzeQ2m2Py2et8blt4HPjv69qB3U4I90cnUBVq3H4p7lAVvz6CaBTyJ+ApMsZbFEi7WIwPNCkfFbBbXcDKBwuWXW0AZPPYLJ0f3dg8t2JVuPL5q1m7dGfZeW8b42+gpYlah86/blUTjCvbA9rj8XiyHC8YcMxTbJT2hMJu8PpddtoopDS1eOHv777PouBz+q7uOkjV+/u39+//cI78U3/t93HHIlH3onN+vPrKcKBN7OcAqrb1m/srh4N2O6sx9QP5H9e9Lp4osVPtx2uL7giXb4ySmfjR03bdIgu76vnQpZdj60RJu0Ek3RmAPYZm5Eu9vamSA6tte3459c/fve36++zpnVkV8kXBsNmSc6QnGgs0XYoMS1ZlKxtL3wqCfdyCYyt2dRM1pKArdod+HnhmjI+MrqbLJw9VPqE75wFt/Gq7YMOAeWqv2Erm71UBzrBt+tFE8pZRvegRT/T8fANIJrmSOWabxmyBjO+TqJoXyK2kXSGxWojwB7WS6NurKg0NBR7MbpoDWxWuZa8w6CVpeP90WNa8woHfN9wRoWyrm6XdT2uG1BaMS5kI7tY+b4x2nu7Wgwubwy7lYuztlkUjS2JpkJZRA1I8MvnRwYmOvlP5wbIS/CvWiuDsqnQmDIor8EYg67VO0TRQ0ReZcPmvjXJBM5CVDZzt7myuIllZDKBPQU0Hb8GlW/6oKLwejFaLh+4yv5lHB8/+xpB4hiTK6yX+IQb8ZvsAH1Wa/6WLSHCLIaEGvOwXkM2flPMCDUGWW5UJCZT3skrZ8dzzfazMtHxFkXirqr3NJzM+UBqVwgtG+ujY17fHEHM59DsjWKc4NBo13OfJjyfG0oYARc28RBzZQ/yyQ15D+cjvuYEo/NYLyI1rqqbZNZdNj8hG0NQcsKVASmMHbdcu7T1SAi9QzvD1h8ZK0Xrx3h63vuEHBYCJ4OhpcMwbMDlVPLm+vu/z24PqcpqTgVmqMZTyxk8pvaQYpeJqHVdTbu+ecHtVT6BaoAz2xtu24ppDN1c6nPQYfWiNCcngqaGRRSNMl45t0H6nL5W97bNUQSDwSRXSsAQbjBrKHkQFW6l8ipw89PyjdAM53a88wbXvhqdaLVo8xakMgRqWNdDwFk/kNop+h0gzujAhutqImym20cNovHcHD+3pAqloJq765Q3SAFJFXu7WpCaUMY1dK1ILWwcMcj6zmyvTu3pM+S0YG3GbNKfWeFkuPHq8fEfH/AcNTeDdoEC1mKsR/VJebQr3zGq2D0EJRoiuo+cxpMHVeMF81IdiaaGx7yZgzbeRWQDsCW/jHN+CSQdmidMsAORg42jP82aHfuzBJnRvuZoGXMr2ncr5zICoRzN70pyIdSW0XwDNtdAfIfL6fFvWr/ZBLAhBWYBlhME2KAuHlnXSGN7EqK/u5CZmyvp8lwuYqFzYN0p/462/7rswBCRsaFvfEHQlFpJJGScWGB6t0mWtY09BFxKXd4nTcHt8LfEjlLiBBl/MSuT1UfJQiMQYKe+sCmNsvZ44xaa6Qqmj6mMg03UNCo2669lxohEIYkVOIUAy6YfVbu8cfP4P2QAydkqHx1yNcpH7zZfrj5i4BWDJ0KGbpuPmQBYltcU8HfJkvaVgDId9TPImwPT5RtJbZoL8SgXhVg0eyHRinRs25hOaMVof2ckPtAvfTlutOurUYG0iF4sbnK0HO5rlcTyRGI40awc27jzsmTsFD3Uz/GMEeCabOaYYtrfTo0vpicf1nWT2lg282vlelNHUd3+8BMmgDm6JaDCfUNtKj345WobSXfc6kJJQ7MtOrTPCaUQGFPGtRuxEK2w7BvtYpa8MOXUTgrvRIVb/kTJzAXaIzmiZ8W4jOrTlZbmWnrgjH4gcd+sdM7tIp658pK+sE1ieojYKVfzUJkIPaQ40iciAaSUKRWYqRhZTHjD7hN4OzHCoCJLc6KGSHWBDDbLXaT2hZ4mNgRT0HMaa2UCShHGijESd6v5JuLifIpkM0BKKqTozBgbneQ/gFZuFpoxMqpppmUFMbNvC0T0HTMR09Nh7Vg891Wk9q6go1GyrOs51Z6ma2mgScVe5VyUmAknVEOK9l2ZLy/S4nn9nb3arpTzemvwUDoqNwJOukuqmKMa5khfSI01EXIh1apJ4GRO+AQq2sheFxtVWgrSXWFBQXvHQ6A1GsXxwvrG36yf0K01smY2NnFzxHa+z5YAJGyat42Wy6CfiiLGZg4RQnYXcsgVgI3Pt3Riw32Rc53wPyCvORssKTQVSrfgW1XaWeK/nn57+vSfp9zbqlScdJIgMjm29AigVRWr86KWTEtHp7KeFMjLxr4byVGzyJeenkw+QSPqC3M8OOBrlEYjlfNw486Oli5QLaezgKmXjbtmQW2qOeahJaxHPPVmyx7aart5N1ZGEeqXCDyYUozWKmeVZ2HeK5DaJW7HIANjSL5Exz2DdXnXtHgGmphLWWi4XUvhvhgBLBet23Wia7a1Sxi+Kk5Rj8WLWtekthjZeEhtNnvfeuxdNw+6voX2L9RC1OPfdUV17fOIuSYpaNdMJyqrBg9vZGkQWQDdJSicXdgZLYu8lgwO8z+t2WJqJT0yaLcki8rcNCJ59kCrs4y57Xdt26qrqXpo1w1Tdphl3Ym1V+OhNRNgqoKU4CWIwU42VMcDFzqnltXOJ+lwBLFqRuNvWU0m3/TOx9CfvN60RPnk6iXGiRrDuNcrAWcpXamcsdmVgfkbNhc4j6P6XK68C8bncLeWJ5T7O+HOc5OKs1Fv271V1ZGgboJNKhT/7JK6XVpgjVy4ojeqUoBBlHNK9EqpJajdmZdlkrF2oNelnRgRFWlLNUSHhmuzh2Gq0lW/r/pKqnQnVCV5l6YkuW7tyo3I4Ilf5FRG77tRRGB0ucCroTNTp4tCkhWXzvy/V3hjdjZ4hp/1SkrDqSiA2I1GrB2N/3E8garpJWtz1MKITP9GOecsG9bUeRUBGLesPZMbJf4KwcuZrl051B1oOX6ccpBNxO0idx5CS63iL3SWCArUUtkx2dqW5DV5JtPNHL0xgIbdnISp4Te7S8EEM4mHF+C8NT4NA9JU/1DWjQ5Wfex6TtmNYi8LjC9SBEf0kNnMvAPTGSwMXHD3CZSXqDUqXWoB5moghOUKMWJ0RPhZ/gH3AF8u/zAghLApWUyd42vl36l9EjWbHggUF6v0gDCeQkh9TvVM9ERHX2vNap+kXU/gnsiKBuIYk2vvdmxUU3ShYpUapkuxpFVoTU/YqSuIU2SUqdbhgKVRXQTjqKwz+OszU3SpJTyDbA670h15JhUru+G8tPPXhQ9SmzwZEdlegGOYiVyb1aqt3wj6QRB6RoQV9dRQWk5pxgreQ6avBerbah41aey5qH70ZNSRaDaYAmmbukJFLW0f+js7hpMY8GQ7Xg1W8p+MSoY6arqBbkNdhaoZGi4Q/G7MRWcGnTYe4wu04NcTehSMXhnxAFTg1rE4VZJImKc3KVzXYZ+ZHzCVCi/DeJfcueqSsP9Dv5M4/jo3eztRy5Zr5Db+aYZ7i7v6mhAjVJDz3XjVurje1XpUUliVGtEF+fgi0EfTJq3O/JWSsVqlnmzqCFvRCilsPBtOmZO3POQx33p0XVo02V7ktnutPIRMEIuhgVLvM29Ibsrkpn2HduC5RrRMZ2SnozNPW6fkKuZBxJuBffY1q1A8WKZyVlDslCJ6TCIdmnSeGC1Umms5YNeYLcdNiiYIRRqqjXd7iYyltRO0uY7hvOgOyS/VNLsHFbTpLIav1yyC3UOTW9h1JIFYKMw00DdvhKhmeRCogGJ269Uy0x4ZKbYlLqyUZ+9eEjFqvIiDxbpLosFxR3YGVlSOF8OyqmlcnSxR7Fvj4qHdGd83g1MxF8P9ydKAYmAblo9daFxPb3wbiAGpKUwTdsOk7lBaWzsgI8yUBHFF6Nw3x2Di4Rynkbzy6ybpiRzxVpaOXUfW2ZkHKdKmhbJp9KJQVW0OP/dyl4yVKBpc/dulA10ZFGO6RvAi7DYTp61aUwOWLfJ+01xCoDIVxm1owDK7CmWJuxEi1Dt/wCjm27Xwgd3UoLWZOCz37puxr7kcUXO1bFsG8bmgdyJXrkFtHd11xurqCNycQ5GKTdg6unuJcKQUycQpjOmuM5zJXrOhQ4hgsyDPOEON4ZGReVrXdgPtU9pImTvn+7lTd/vXa/naygOSG2FcsXb2VmqBJX1rC/XHhlodtpxJqNCJPqd5t3lCs/Cc2xl+jcWj9XtIs6aHtFVhaYsPeVLPtIBUU91eWqvdQDulLC+OmRCVh5f87TLMW192Bb1AEknpTSlKqbGvM4FBYoSpbjNJkO0Gp5M0LKCp6yHPvhuuyNSRHL9V0KsTa7dt2uRtp1ZPnlNgvP5rEvfU6f7273X6t28aai7B+S3YJax6PidjVoOutmWLZV9g0XfcEE+5mAmTZcK4cnqh5aH6d49qafaQq8kTLrTtrgID3vUsntk6scfpOUVro+aXhXeg6UbnXgiboRw3j20bCTwxsknNfmrz2tSAv9YAZhK2R7DSuz1d628SB85RZrWbXICxudxO1Xsd5/BGvDLjd8WbPp+qQm4xX/SSnmkTNm1x073iUpWkM1LAM7GRXALJmBpHq6SxO95FIt1LDAzq9dI0cPamew3QNP6mdZLcIDk7mgWpJ176ezXNWkoCRYJ6M59Pjq/8S705gnebXhgTTC9L2zaPMQz91LdIZUiS546Gm7Zuw8Dr+v5EHWkduCNJ/8JzZlMQJcDMBvKqu4xMqXPThQO9gDjEQoSjBIdX5aJCfX0TefZjghWbNsyMrrzQ+ahTIpGpIeJ3TSbmBq2cUrnyEliCUui6x7XoOuctaBpO1hpeha7dEiYHJD6GnksQGyZkwdWuSoNs5bPK+nYz6dic6nz54tdzb8sAh95cO3IRZCaPhiqsHAgoebm0B5JmRoMc+b/ymRj1x18NsXsYv081jHguGJfNp56GPdtO/pxftrJnfyWQ08OPIJip5da5qhOaFyNjMPUq5JYB5aR1zCTxJ3RHIdLuff8UeEBwz8BwtjMnhuCruVOy6RPahkexzVjK0dOSKHorzi/r7nmll+t07zaTcTTXVJ8aaddNxo33tXvZiQprjC0/2IkdPrD0TmcZlmxrLdVxuCoWxbeAuqBKeQNZng/r5fcBUA2ozasNS179KsL2XRMxohLjSPV4nRF3MpIG2Og6/ywD1xAmMr8rRhF6GOI33VTVPlB+DQSatinYI9DI+U3mRaCjO3fvQYjHDpv5RWgz40woTS2523x+t3LMedTXDo8/4+Jc9Eh36Z3TgIUO3TuizZACVQvei0858O+H2D9FWwAB/3cr3a7WhpwaRhz0mHBgUSdhmbPdIyY6+e31WkaURyC79KoAuGp849L9pGkbF/4UH5x378UzYWvEB4RUXtrdT7eYl4YrKN/G++DQI/i1K8GHJx/FwI/R1QjsE9HT9bdH5DFrHXd4Rl2Qyio08k5a2m+SFrA84gGcKtn8cMwM8BLD9t5CeWQNbcS3Ppsb5M9hgW4wxZ2mxhWInTj2cvbU/27KwNnVpzfebGlLehfi5QKkxh+OIdM7X02PQiS2ln7Loj0TU+zbIvGWurL3am5emxq2vZd/c4SGIOy29dce21nWuZmiEHUmjJiZ2/r7RHU8R/5pBp32PAtmLqe3erFprJM41Jw370I3Wbxa+/YpCM2phdKyTDVdGS0t7X+miMJRfQejTdeRWA9+zlENdae33Rz2bh6ehl8wlN/Mv7JkM28JLdczm58vMWr3G0WmQWPO1vcdb3qy5MJ2tNcPLzrm5IdIShPFUY02HNLaxd21/RcUD0dq"
}      # 備援資料（壓縮後的文字），網路或 GitHub 出問題時才會用到
_TRUTH = 'eNp1Vu2O4jAM/H/PYq0S56PJW9wboC5k2eqgIIpO7NvfJE17/TBaiSKWscfjsc29fXapfx66E7XnREN60fE7Dc/Dve16euBd158Pn3d8ervQ1+dQPjuk45mu7evw/aD0avsz3S6ne2r/0HC53RMdW3p+txc6dUNqh/Trt1KOrKavdG0vifpb//y5d0d8QxtFrCzhlXTDeOoPSxrP/vbAl/HI6EjeUMG2w8/1/rxdwfs4ol1F26ai8R4s/6bH0H0C8ZMGhGBN3o4h5twaOK0rOmfVH4F4kxsprJJyA619M6J1RvOHKuhdboPia26UDrm6firdGAAQqIm5aKF048mylB6kmH1Fh/xE+i0aaV3cZ3a5cC7f1m6Uze+wzpFXb0WvvKts5XVfuAu562+0K90GCbckvw8RyQnasSNuTA3AJYAjgxDLCiCLW5puRkM7o0Jt/Gg6VypYoSPZQKJhrRuxQZXMvDNNQGYjZWYYTPtqGrcovOLiArdRHUhTre7Vm35HJs8y3me7TqOSM3vwNvjbSq6VIZhCDAL6Rk2yx5nESjidp1XsOgM3Ui8t0GRgO2lasz9MFI2T9YtziNw5FodOY7C9lqtA+UbVybW+GFieXM0mO2jXRZVnjysLUwLsPJC1mpy7Mz+bWgP7sjiCLINhKX3EiuNmxLuyMWoTFvjCIO8d2YVgq+sEeyd7OKvkglA7Jq8oz3N2vxWv4GFk30jZ4Xvnl07WbyIA7ljYm1CLJ/b/TbgiH/Q8vBsPQrxmSu7r4hB7H1abZ55/jBiHOgLRLUZgFwD/9UGUf3F3yhLQlcOiAlZY/E5c/BhbFeebN7afhUFmGO3dNsgXwNqFh8bDOVMYA0AsJ95dyM9hHIDo5tPHewqwIMILGvh893VtopE3CeP4ySOUDRymNerLJuFCYEXfwD/+3QULeW4BV+P8ejKCCfKc2ij+bsk+mvhz5b/tIdrjRBfmBWIm+UcXatFEDAdJFxx7g7ELywUfr0IRAAuve6VTxTZuxq54K7viHYoQK+WimpVbFp2Na9Vm8syq6H8Dlu4M'            # 檢查清理結果時用的「標準答案」（只有檢查器會用到）
BANDS = {'age': [35.0, 71.0], 'resting_bp': [100.0, 180.0], 'chol': [149.0, 406.7], 'max_hr': [95.0, 192.0], 'oldpeak': [0.0, 4.2], 'ca': [0.0, 3.0]}             # 各欄位「合理範圍」（缺值要補的值，落在這裡面都算合理）

BASE_URLS = [f"https://raw.githubusercontent.com/{REPO}/v1.0/data/",      # 先試固定版本（tag）
             f"https://raw.githubusercontent.com/{REPO}/main/data/"]       # 再試 main 分支

def load_csv(name):
    # 全部以文字讀入，避免 pandas 偷偷幫你改資料（例如把 NA 變成空值）
    text, src = None, None
    for base in BASE_URLS:
        try:
            with urllib.request.urlopen(base + name, timeout=10) as r:
                text, src = r.read().decode("utf-8"), "GitHub"
            break
        except Exception:
            continue
    if text is None:
        text, src = zlib.decompress(base64.b64decode(_EMBEDDED[name])).decode("utf-8"), "內嵌備援（GitHub 讀不到）"
    print(f"  {name}：從「{src}」載入")
    return pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False)

df_dirty_small = load_csv("heart_dirty_small.csv")    # 40 位病人：給你手動清理
df_dirty_full  = load_csv("heart_dirty_full.csv")     # 全部病人：等一下讓程式自動清理
df_truth_small = pd.read_csv(io.StringIO(zlib.decompress(base64.b64decode(_TRUTH)).decode("utf-8")), dtype=str, keep_default_na=False)

print(f"\n練習用小資料：40 位病人（其中 2 位重複出現，所以共 {len(df_dirty_small)} 列）；完整資料：{len(df_dirty_full)} 列")
print("👀 這就是等一下要清理的資料（␣＝多餘的空格，⟨空白⟩＝空的格子）：")
show(df_dirty_small)

---
## 單元 1｜資料如何才能「餵進去」

AI 模型只吃**數字**。先認識兩個名詞：

In [ ]:
#@title 圖｜什麼是特徵與標籤
show_fig('features_labels')

一位病人＝一列　一個欄位＝一個特徵　想預測的答案＝標籤 特徵 features（X） 我們「給模型看」的資料 age 63 67 41 sex male male female chest_pain typical nonanginal nontypical resting_bp 145 160 130 chol 233 286 204 max_hr 150 108 172 disease no yes no 標籤 label（y）：想預測的答案 模型的工作：看 X → 猜 y （像實習醫師看過很多病歷，學著判斷診斷）

**特徵（feature）**是我們「給模型看」的資料（一欄一個）；**標籤（label）**是我們希望它「預測」的答案。這裡的標籤是 `disease`（有沒有冠心病）。

那麼，**為什麼電腦會拒絕吃髒資料？**

In [ ]:
#@title 圖｜為什麼電腦拒絕吃髒資料
show_fig('why_dirty')

為什麼電腦「拒絕」吃髒資料？ 原始資料 （各系統匯出） 轉成數字 模型 （只吃數字） 文字混在數字裡 "54y"　"110 mmHg" → 轉不成數字 → NaN 空白、?、NA、unknown "" "?" "NA" → 全部變成 NaN 轉得成，但是錯！ 膽固醇 5.46（mmol/L） 被當成 5.46 mg/dL "Male" 被當成 0 → 不會報錯 NaN 會「傳染」：1 ＋ NaN ＝ NaN　→　loss ＝ NaN　→　整個訓練壞掉 更可怕的是第三種：沒有任何警告，模型卻學到錯的東西

除了「轉不成數字」，還有一種最常見的資料問題：**缺值**。遇到缺值，第一個問題不是「怎麼補」，而是：**為什麼缺？**

In [ ]:
#@title 圖｜為什麼缺？怎麼處理？
show_fig('missing')

為什麼缺？缺值本身，可能就是資訊 ① 隨機缺 例：檢體溶血、系統漏存、單純忘了填 缺的原因跟病情無關 → 刪除或填補通常比較安全 ② 因為「沒做這項檢查」而缺 例：胸痛病人沒驗心肌酵素（troponin） 因為醫師覺得不像心肌梗塞 「沒做」本身就帶有臨床資訊 → 隨便填補，會把這個訊號抹掉 處理方式（只講概念） 刪除 整列或整欄拿掉 簡單 但損失資料；如果缺值 有規律，會造成偏誤 填補 中位數、眾數、預測值… 保留資料 但填進去的是「猜的」 （要記得補值用的數字 也只能從訓練資料算） 標記 多加一欄「是否缺值」 讓模型自己學 「缺」代表什麼 （特別適合第②種）

**處理缺值的三個概念**：**刪除**（拿掉整列或整欄）、**填補**（用中位數、眾數等補上）、**標記**（多加一欄「是否缺值」）。
今天的練習會用到「刪除」與「填補」。

### 資料字典


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">欄位</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">意義</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">合理範圍／標準寫法</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>patient_id</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">病人編號</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P001…；<b>每位病人只應出現一次</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>age</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">年齡（歲）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">18–100</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>sex</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">性別</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>male</code> / <code>female</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>chest_pain</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">胸痛類型</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>typical</code>（典型心絞痛）、<code>nontypical</code>（非典型）、<code>nonanginal</code>（非心絞痛）、<code>asymptomatic</code>（無症狀）</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>resting_bp</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">休息時血壓（mmHg）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">70–250，<b>只要數字</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>chol</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">血清膽固醇（<b>mg/dL</b>）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">100–600，<b>只要數字</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>fbs</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">空腹血糖 > 120 mg/dL？</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>1</code>＝是、<code>0</code>＝否</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>rest_ecg</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">休息心電圖</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>0</code> 正常、<code>1</code> ST-T 異常、<code>2</code> 左心室肥厚</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>max_hr</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">運動測試最大心跳（bpm）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">60–220，<b>只要數字</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>exang</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">運動誘發心絞痛？</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>1</code>＝是、<code>0</code>＝否</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>oldpeak</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">運動後 ST 段下降（mm）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">0–8</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>slope</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">運動高峰 ST 段斜率</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>1</code> 上升、<code>2</code> 平、<code>3</code> 下降</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>ca</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">透視下有鈣化的主要血管數</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">0–3</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>thal</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">鉈（thallium）心肌灌注掃描</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>normal</code> / <code>fixed</code> / <code>reversible</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>disease</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>標籤</b>：有冠心病嗎？</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>yes</code> / <code>no</code></td></tr></table>


> 你不需要懂這些指標的臨床意義，只需要知道**每一欄「應該長什麼樣子」**。

### 資料要怎麼變成數字？（等一下的程式就是照這張表做）


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">欄位</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">轉換規則</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">例子</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">數字欄位（<code>age</code>、<code>chol</code>…）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">直接當數字讀；<b>讀不了 → NaN</b>（Not a Number，「不是數字」）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>"54"</code> → 54　<code>"54y"</code> → <b>NaN</b>　<code>"?"</code> → <b>NaN</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>sex</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">剛好等於 <code>male</code> → 1；<b>其他一律 → 0</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>"male"</code> → 1　<code>"female"</code> → 0　<code>"Male"</code> → <b>0（錯了！）</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>disease</code>（標籤）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">剛好等於 <code>yes</code> → 1；<b>其他一律 → 0</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>"yes"</code> → 1　<code>"no"</code> → 0　<code>"Y"</code> → <b>0（錯了！）</b></td></tr></table>


`chest_pain`、`thal` 是文字類別，單元 3 才會處理；單元 1、2 的模型先不使用它們。

In [ ]:
#@title 1-1　👀 大家來找碴：這幾位病人的資料，哪裡怪怪的？
DEMO_IDS = ['P005', 'P110', 'P021', 'P111', 'P135', 'P145', 'P058', 'P079', 'P189', 'P247', 'P128']
demo_rows = df_dirty_small[df_dirty_small["patient_id"].isin(DEMO_IDS)]
print(f"從 {len(df_dirty_small)} 列裡挑出 {len(demo_rows)} 列。給你 1 分鐘，找出『哪一格』怪怪的。")
show(demo_rows)

In [ ]:
#@title 1-2　🔍 公布答案：哪些格子有問題？
show_issues(df_dirty_small, rows=list(demo_rows.index))

In [ ]:
#@title 1-3　把資料「硬轉」成數字：到底發生什麼事？
d_bad = df_dirty_small.copy()

# 規則 1：數字欄位 → 直接當數字讀；讀不了的格子，變成 NaN
for c in ALL_NUMERIC:
    d_bad[c] = pd.to_numeric(d_bad[c], errors="coerce")
# 規則 2：sex → 剛好等於 "male" 的變成 1，其他『全部』變成 0
d_bad["male"] = (df_dirty_small["sex"] == "male").astype(int)
# 規則 3：標籤 disease → 剛好等於 "yes" 的變成 1，其他『全部』變成 0
d_bad["label"] = (df_dirty_small["disease"] == "yes").astype(int)

rule_rows = []
for c in ALL_NUMERIC:
    rule_rows.append({"欄位": c, "轉換規則": "當數字讀", "轉換後 NaN 的個數": int(d_bad[c].isna().sum())})
rule_rows.append({"欄位": "sex", "轉換規則": "male → 1，其他 → 0", "轉換後 NaN 的個數": 0})
rule_rows.append({"欄位": "disease", "轉換規則": "yes → 1，其他 → 0", "轉換後 NaN 的個數": 0})
rule_rows.append({"欄位": "chest_pain, thal", "轉換規則": "單元 1 的模型不用", "轉換後 NaN 的個數": "—"})
show(pd.DataFrame(rule_rows))
print("👉 sex 和 disease 的 NaN 是 0，不是因為它們沒問題，而是規則 2、3 把『不認得的寫法』全部當成 0，完全不會報錯。")

X_bad = add_male(d_bad)[FEATURES_V1].values.astype("float32")
y_bad = d_bad["label"].values.astype("float32")
_, h_bad = train_mlp(X_bad, y_bad, epochs=10)
print("\n帶著 NaN 去訓練 10 個 epoch，每一步的 loss（模型的『錯誤程度』）：")
print([round(v, 3) for v in h_bad["train"]])
print("👉 loss 變成 nan：模型一碰到 NaN，整個訓練就壞掉了，而且不會告訴你為什麼。")

In [ ]:
#@title 1-4　👀 再來找碴：sex 與 disease 轉成數字之後，對嗎？
CONV_IDS = ['P005', 'P030', 'P110', 'P167', 'P135', 'P247', 'P132', 'P109', 'P044', 'P036']
conv = df_dirty_small[df_dirty_small["patient_id"].isin(CONV_IDS)][["patient_id", "sex", "disease"]].copy()
conv["male"] = (conv["sex"] == "male").astype(int)            # 規則 2
conv["label"] = (conv["disease"] == "yes").astype(int)        # 規則 3
conv = conv[["patient_id", "sex", "male", "disease", "label"]]
conv.columns = ["patient_id", "sex（轉換前）", "male（轉換後）", "disease（轉換前）", "label（轉換後）"]
print("每一列：左邊是原本的寫法，右邊是程式轉出來的數字（male：1＝男、0＝女；label：1＝有病、0＝沒病）。")
print("轉換『有成功』（沒有 NaN），但哪些轉換結果是錯的？")
show(conv)

In [ ]:
#@title 1-5　🔍 公布答案：哪些轉換「默默」出錯？
true_male  = conv["sex（轉換前）"].str.strip().str.lower().isin(["m", "male"]).astype(int)
true_label = conv["disease（轉換前）"].str.strip().str.lower().isin(["yes", "y", "1"]).astype(int)
bad_sex   = conv["male（轉換後）"] != true_male
bad_label = conv["label（轉換後）"] != true_label

def paint(_):
    css = pd.DataFrame("", index=conv.index, columns=conv.columns)
    css.loc[bad_sex,   ["sex（轉換前）", "male（轉換後）"]] = "background-color:#ff8a80 !important;color:#000;font-weight:bold"
    css.loc[bad_label, ["disease（轉換前）", "label（轉換後）"]] = "background-color:#ff8a80 !important;color:#000;font-weight:bold"
    return css
show(conv, styler=conv.style.apply(paint, axis=None))

all_true_male  = df_dirty_small["sex"].str.strip().str.lower().isin(["m", "male"])
all_true_label = df_dirty_small["disease"].str.strip().str.lower().isin(["yes", "y", "1"])
n_sex_bad   = int((all_true_male != (df_dirty_small["sex"] == "male")).sum())
n_label_bad = int((all_true_label != (df_dirty_small["disease"] == "yes")).sum())
print(f"全部 {len(df_dirty_small)} 列裡：{n_sex_bad} 位的性別被默默轉錯；{n_label_bad} 位『有病』的病人被默默當成『沒病』。")
print("👉 有些列是『運氣好剛好對』（例如 'female ' 多了空格，轉成 0，剛好還是女性）。")
print("   標籤被轉錯，等於你教模型的答案是錯的。這種錯誤沒有任何警告，最危險。")

In [ ]:
#@title 1-6　建立練習用的 Google 試算表（第一次會跳出授權視窗）
MODE = "sheets"          # "sheets"＝Google 試算表（建議）；"csv"＝下載 CSV 編輯後再上傳（備案，見下方說明）
SHEET_TITLE = "heart_dirty_practice"
CLEANED_CSV = os.path.join(WORKDIR, "heart_cleaned_small.csv")      # csv 模式：請把清理好的檔案用這個名字存到 Drive 資料夾

df_dirty_small.to_csv(os.path.join(WORKDIR, "heart_dirty_small.csv"), index=False)   # 備案用：Drive 資料夾裡也放一份

def _to_cell(v):
    # 數字寫成數字，其他維持文字（試算表裡看起來才像真的資料）
    try:
        f = float(v)
        return int(f) if (f.is_integer() and "." not in v) else f
    except (ValueError, TypeError):
        return v

if MODE == "sheets" and not IN_COLAB:
    print("（不在 Colab 環境，自動改用 csv 模式）"); MODE = "csv"

if MODE == "sheets":
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    gc = gspread.authorize(default()[0])
    try:
        sh = gc.open(SHEET_TITLE)
        print("✅ 找到先前建立的試算表，你之前的修改都還在。")
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_TITLE)
        rows = [list(df_dirty_small.columns)] + [[_to_cell(v) for v in r] for r in df_dirty_small.values.tolist()]
        sh.sheet1.update(values=rows, range_name="A1")
        print("✅ 已建立新的試算表。")
    print("\n👉 請點開這個連結，開始清理（也可以在 Google 雲端硬碟找到它）：")
    print("  ", sh.url)
    print("\n（想整個重來：把這個試算表丟進垃圾桶，再重新執行這一格）")
else:
    print("csv 模式：")
    print(f"  1. 到 Google 雲端硬碟 → AI_preprocessing_class 資料夾 → 找到 heart_dirty_small.csv")
    print( "  2. 用 Google 試算表（或 Excel）開啟，修改")
    print( "  3. 另存／下載成 CSV，檔名改成 heart_cleaned_small.csv，上傳回同一個資料夾")

### 👉 你的任務（約 15 分鐘，兩人一組）

打開試算表，把這 40 位病人的資料**清理到可以餵給模型**。每種問題各有幾個，找出來並修好：


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">問題類型</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">怎麼辨認</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">可以怎麼處理</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">缺值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">空白、<code>?</code>、<code>NA</code>、<code>unknown</code>、<code>N/A</code>…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">補一個<b>合理的值</b>（例如該欄中位數），或刪掉整列</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">不可能的數值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">超出上面表格的合理範圍</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">能判斷是打錯就改；不能判斷就當缺值，補合理的值</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">單位不一致</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">膽固醇應該是 100–600；出現 5.46 可能是 mmol/L</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">mg/dL ＝ mmol/L × 38.67</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">數字欄位混入文字</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>110 mmHg</code>、<code>143bpm</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">只留下數字</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法不一</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>M</code>/<code>Male</code>、<code>ASYMPTOMATIC</code>、<code>reversable</code>、<code>1</code>/<code>Y</code>…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">改成上面表格的<b>標準寫法</b>（小寫、沒有多餘空格）</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">重複列</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">同一個 <code>patient_id</code> 出現兩次</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">只留一列</td></tr></table>


**小技巧**：`Ctrl+H`（尋找與取代）、「資料 → 資料清理 → 移除重複項目」、點欄位標題排序，會讓你快很多。
**規則**：不要改標題列（第 1 列）的欄位名稱。

⚠️ **檢查器會看三件事**：① 有問題的格子修好了沒；② **沒有問題的格子被你改掉了沒**（改壞正常資料是最糟的）；③ 有『固定答案』的格子（拼字、單位換算、去掉文字）答案要對，缺值與離譜的值則只要填『合理』的值。
**想一想**：遇到「看起來很誇張」的數值，你要怎麼判斷它是**打錯**還是**真的**？刪掉整位病人，有沒有代價？

修完之後，執行下面兩格：先檢查，再訓練。檢查會告訴你還剩什麼問題；卡住就把 `HINT` 調大一點。

**🎓 老師備註（單元 1）**
- 1-1 挑的 12 列涵蓋所有錯誤類型（缺值、不可能值、單位、文字混入、類別寫法、重複）。給 1 分鐘讓學生喊，再用 1-2 公布。
- 1-3：先回答「為什麼 sex 沒有 NaN」——規則是 `== "male"`，不認得就當 0，不會報錯。1-4／1-5 讓學生親眼看到「轉換成功」≠「轉換正確」；`'female '` 是運氣好剛好對。
- 40 列裡共注入：缺值 5＋原本就缺 2、不可能值 4、單位錯誤 3、文字混入 4、類別寫法 11、重複 2。完整答案：`teacher/answer_key_small.csv`。
- **陷阱題（刻意保留）**：`P153` 的 chol = 564、`P092` 的 oldpeak = 6.2，都是**真實**極端值，不是錯誤。若學生改掉，檢查器會報「不該改的被改了」，正好討論「由有臨床判斷力的人決定」。
- 原本就缺值的是 `P088`（thal）與 `P167`（ca）：不是我們造的，是資料集本來就有。
- 學生卡住或你想直接示範：執行下一格「老師捷徑」，等於一位完美的學生。

**🎓 老師備註：小資料集（40 位）的注入錯誤完整答案**（另外 P088 的 thal、P167 的 ca 是資料集本來就缺）


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">patient_id</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">欄位</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">類型</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">髒值</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">原本的值</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P111</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chol</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">缺值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">``</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>307</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P268</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">ca</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">缺值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>?</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>1</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P224</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">thal</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">缺值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">``</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>reversible</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P135</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">resting_bp</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">缺值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>unknown</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>122</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P145</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">oldpeak</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">缺值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">``</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>0.6</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P110</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">age</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">不可能值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>520</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>39</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P247</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">resting_bp</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">不可能值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>0</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>100</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P079</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chol</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">不可能值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>0</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>245</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P189</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">max_hr</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">不可能值</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>1950</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>195</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P059</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chol</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">單位錯誤</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>7.06</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>273</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P021</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chol</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">單位錯誤</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>5.46</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>211</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P078</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chol</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">單位錯誤</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>7.96</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>308</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P109</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">resting_bp</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">數字欄位混入文字</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>120 mmHg</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>120</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P058</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">resting_bp</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">數字欄位混入文字</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>110 mmHg</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>110</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P145</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chol</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">數字欄位混入文字</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>240 mg/dL</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>240</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P123</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">max_hr</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">數字欄位混入文字</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>143bpm</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>143</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P135</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">sex</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>female </code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>female</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P167</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">sex</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>MALE</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>male</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P110</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">sex</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>m</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>male</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P009</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chest_pain</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>ASYMPTOMATIC</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>asymptomatic</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P110</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chest_pain</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>asymptomatic </code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>asymptomatic</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P153</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">chest_pain</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>Nonanginal</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>nonanginal</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P111</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">thal</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>reversible defect</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>reversible</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P236</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">thal</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>NORMAL</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>normal</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P247</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">disease</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>Yes</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>yes</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P132</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">disease</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>0</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>no</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P109</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">disease</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">類別寫法</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>y</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>yes</code></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P210</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">(整列)</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">重複列</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>duplicate row</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">``</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P128</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">(整列)</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">重複列</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><code>duplicate row</code></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">``</td></tr></table>


In [ ]:
#@title [老師] 捷徑：模擬一位把資料清得很完美的學生（學生版沒有這一格）
_perfect = auto_clean(df_dirty_small)
if MODE == "csv":
    _perfect.to_csv(CLEANED_CSV, index=False)
else:
    sh.sheet1.clear()
    sh.sheet1.update(values=[list(_perfect.columns)] + _perfect.values.tolist(), range_name="A1")
print("✅ 已用程式清理並寫回（相當於一位完美學生）。")

In [ ]:
#@title 1-7　讀回你清理的結果並檢查（可以重複執行，直到全部通過）
HINT = 1     # 1：只告訴你還有幾類問題　2：再告訴你是哪幾個欄位　3：連 patient_id 都告訴你　4：連答案都告訴你

def read_student_table():
    if MODE == "sheets":
        rows = sh.sheet1.get_all_values()
        d = pd.DataFrame(rows[1:], columns=rows[0])
    else:
        d = pd.read_csv(CLEANED_CSV, dtype=str, keep_default_na=False)
    return d[d.apply(lambda r: "".join(map(str, r)).strip() != "", axis=1)].reset_index(drop=True)   # 去掉全空白的列

df_student = read_student_table()
print(f"讀到 {len(df_student)} 列（原本 {len(df_dirty_small)} 列）")
all_ok = check_cleaned(df_student, hint=HINT, dirty=df_dirty_small, truth=df_truth_small, bands=BANDS)

In [ ]:
#@title 圖｜我們的超小神經網路長什麼樣
show_fig('tiny_net')

我們的超小神經網路：只有 4 個感知器 age male resting_bp chol fbs rest_ecg max_hr exang oldpeak slope ca 感知器 1 tanh 感知器 2 tanh 感知器 3 tanh 感知器 4 sigmoid 有病的機率 （0 到 1） 輸入：11 個特徵 隱藏層：3 個感知器 輸出層：1 個 每個感知器：Σ（輸入 × 權重）＋ 偏差 → 激活函數 參數：11×3＋3＋3＋1 ＝ 40 個（單元 4 的 22 個輸入則是 73 個）

In [ ]:
#@title 1-8　用你清好的資料，訓練一個超小的神經網路（輸入 → 3 個感知器 → 1 個感知器，共 4 個）
def train_on_table(df, epochs=300):
    try:
        X, y = to_xy(df)
    except Exception as e:
        print("❌ 資料還沒辦法餵進模型。Python 的錯誤訊息是：\n  ", e)
        print("   → 回到 1-7 看檢查結果；通常是數字欄位裡還有文字或空白。")
        return None
    if np.isnan(X).any():
        print("⚠️ 資料裡還有 NaN，loss 會變成 nan。")
    model, hist = train_mlp(X, y, epochs=epochs)
    plt.figure(figsize=(8, 4.5)); plt.plot(hist["train"])
    plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("Training loss"); plt.tight_layout(); plt.show()
    acc = evaluate(model, X, y)["acc"]
    print(f"訓練 {epochs} 個 epoch：loss {hist['train'][0]:.3f} → {hist['train'][-1]:.3f}；"
          f"在這 {len(y)} 位病人上的準確率 {acc:.1%}")
    print("（to_xy 會先把每個欄位縮到大約 0–1，小網路才學得起來；單元 3 會解釋為什麼。）")
    return model, hist

result_student = train_on_table(df_student)

### 同一件事，讓程式來做

你剛剛花了十幾分鐘，只清理了 **40 列**。真實世界的資料動輒幾萬、幾百萬列，不可能手動。
但請注意：程式做的，其實就是你剛剛做的每一個判斷，只是把它們**寫成規則**——所以**你得先懂資料有哪些問題**，程式才寫得出來。

下面把規則一條一條寫出來，每一步都列出「改了哪些格子」。

In [ ]:
#@title 1-9　程式清理（一）：去空格、統一「缺值」寫法、移除重複
t_start = time.time()
d_full = df_dirty_full.copy()                           # 從原始資料重新開始（可以重複執行）

# 規則 1：每個格子前後的空格都去掉（"female " → "female"）
before = d_full.copy()
d_full = d_full.apply(lambda s: s.str.strip())
show_changes(before, d_full, "規則 1：去掉前後空格")

# 規則 2：這些寫法都代表「缺值」→ 統一變成 NaN
MISSING = ["", "?", "NA", "N/A", "na", "n/a", "null", "unknown", "-", "--"]
before = d_full.copy()
d_full = d_full.replace({t: np.nan for t in MISSING})
show_changes(before, d_full, "規則 2：缺值的各種寫法 → NaN")

# 規則 3：同一個 patient_id 只留第一列
dups = d_full[d_full.duplicated("patient_id", keep=False)].sort_values("patient_id")
print(f"▶ 規則 3：重複的病人（共 {len(dups)} 列，只保留每位的第一列）")
show(dups.head(6))
d_full = d_full.drop_duplicates(subset="patient_id", keep="first")
print(f"目前 {len(d_full)} 列")

In [ ]:
#@title 1-10　程式清理（二）：抽出數字、單位換算、不可能的數值
# 規則 4：數字欄位只留下「數字」部分（"120 mmHg" → 120）
before = d_full.copy()
for c in ALL_NUMERIC:
    d_full[c] = pd.to_numeric(d_full[c].str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce")
show_changes(before, d_full, "規則 4：抽出數字（去掉 mmHg、bpm…）")

# 規則 5：膽固醇小於 30 的，其實是 mmol/L → 乘 38.67 換成 mg/dL
before = d_full.copy()
d_full.loc[d_full["chol"] < 30, "chol"] *= 38.67
show_changes(before, d_full, "規則 5：膽固醇單位換算", cols=["chol"])

# 規則 6：超出合理範圍的 → 當成缺值（範圍就是上面資料字典那張表）
print("合理範圍：", {c: RANGES[c] for c in ["age", "resting_bp", "chol", "max_hr", "oldpeak"]})
before = d_full.copy()
for c, (lo, hi) in RANGES.items():
    d_full.loc[(d_full[c] < lo) | (d_full[c] > hi), c] = np.nan
show_changes(before, d_full, "規則 6：不可能的數值 → NaN")

In [ ]:
#@title 1-11　程式清理（三）：把文字類別的寫法統一
# 規則 7：先轉小寫，再用「對照表」把各種寫法對到標準寫法
FIX = {
    "sex": {"m": "male", "male": "male", "f": "female", "female": "female"},
    "chest_pain": {"typical": "typical", "typical angina": "typical",
                   "nontypical": "nontypical", "non-typical": "nontypical", "atypical": "nontypical",
                   "nonanginal": "nonanginal", "non-anginal": "nonanginal", "non anginal": "nonanginal",
                   "asymptomatic": "asymptomatic"},
    "thal": {"normal": "normal", "fixed": "fixed", "fixed defect": "fixed",
             "reversible": "reversible", "reversable": "reversible", "reversible defect": "reversible"},
    "disease": {"yes": "yes", "y": "yes", "1": "yes", "no": "no", "n": "no", "0": "no"},
}
print("對照表（例）：", {k: FIX["sex"][k] for k in ["m", "male", "f"]}, "…")
before = d_full.copy()
for c, m in FIX.items():
    d_full[c] = d_full[c].str.lower().map(m)
show_changes(before, d_full, "規則 7：類別寫法統一（大小寫、縮寫、拼字）", cols=list(FIX))
print("對照表裡找不到的寫法會變成 NaN；目前這四欄的 NaN 個數：", d_full[list(FIX)].isna().sum().to_dict())

In [ ]:
#@title 1-12　程式清理（四）：補上剩下的缺值，存檔，檢查，訓練
# 規則 8：剩下的缺值：數字欄位用「中位數」補；類別欄位用「眾數」（最常出現的）補
fill_values = {c: d_full[c].median() for c in ALL_NUMERIC}
fill_values.update({c: d_full[c].mode()[0] for c in CAT_COLS})
print("要補的值（每一欄的中位數／眾數）：")
show(pd.DataFrame({"欄位": list(fill_values), "補上的值": list(fill_values.values())}))
before = d_full.copy()
for c in ALL_NUMERIC + list(CAT_COLS):
    d_full[c] = d_full[c].fillna(fill_values[c])
show_changes(before, d_full, "規則 8：補缺值")

d_full["chol"] = d_full["chol"].round(0)
d_full[ALL_NUMERIC] = d_full[ALL_NUMERIC].astype(float)
df_clean = d_full.reset_index(drop=True)
elapsed = time.time() - t_start
print(f"\n原始 {len(df_dirty_full)} 列 → 清理後 {len(df_clean)} 列（移除 {len(df_dirty_full) - len(df_clean)} 列重複），8 條規則全部跑完只花了 {elapsed:.2f} 秒")
check_cleaned(df_clean)
df_clean.to_csv(os.path.join(WORKDIR, "heart_clean_full.csv"), index=False)
show(df_clean.head())
result_full = train_on_table(df_clean)

In [ ]:
#@title [老師] 確認：上面八條規則與工具箱的 auto_clean 結果完全一致（學生版沒有這一格）
assert df_clean.equals(auto_clean(df_dirty_full)), "1-9～1-12 與工具箱 auto_clean 結果不同！"
print("✅ 一致")

### 先打個問號 ❓

剛剛顯示的準確率，是模型在**它自己學過的資料**上算出來的。
**這個數字可信嗎？** 就像學生拿「考古題」當考試成績——這就是單元 2 要談的事。

**🎓 老師備註**：自動清理用「中位數／眾數」補缺值，且是用**全部資料**算的，這本身就是輕微的資訊洩漏；單元 4 會示範正確做法：補值用的數字也要「只用訓練資料算、存起來、推論時重複使用」。

---
## 單元 2｜資料怎麼分配

模型的目的，是在**沒看過的新病人**身上也準。所以我們不能用「它學過的資料」來評分。
把資料分成三份，就像準備考試：


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">資料</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">像是</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">用途</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">可以用幾次</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>訓練集 train</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">課本與考古題</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">模型從這裡學習</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">無限次</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>驗證集 validation</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">模擬考</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">比較不同做法、調整設定</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">很多次（但用越多，越會偏向它）</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>測試集 test</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>正式考試</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">最後給模型一個誠實的成績</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>只有一次</b></td></tr></table>


### 什麼是「隨機種子」（random seed）？

電腦的「隨機」其實是**算出來的**，就像**洗撲克牌**：「怎麼起手洗牌」就是**種子**。


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">種子</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">結果</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>同一個種子</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">每次洗出一模一樣的牌序 → 切出來的訓練／測試資料也一模一樣（別人可以<b>重現</b>你的結果）</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>換一個種子</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">洗出另一種牌序 → 切出來的人不同，成績也跟著不同</td></tr></table>


程式裡，切分資料的種子叫 `random_state`，訓練模型的叫 `seed`。

In [ ]:
#@title 2-1　隨機切成「訓練 80％／測試 20％」，換 5 個隨機種子看看
if "df_clean" not in globals():                 # 保險：沒跑單元 1 也能繼續
    df_clean = auto_clean(df_dirty_full)
X_all, y_all = to_xy(df_clean)

TEST_SIZE = 0.2        # 👈 試試看：改成 0.5、0.9 再重新執行，成績會怎麼變？（test_size＝拿多少比例當測試集）
accs = []
for seed in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=TEST_SIZE, random_state=seed)
    model, _ = train_mlp(Xtr, ytr)
    acc = evaluate(model, Xte, yte)["acc"]
    accs.append(acc)
    print(f"random_state={seed}：測試集準確率 {acc:.1%}　（測試集只有 {len(yte)} 位病人）")

plt.figure(figsize=(7.5, 4.2)); plt.plot(range(5), accs, "o-"); plt.ylim(0.4, 1)
plt.xlabel("random_state (how the data was split)"); plt.ylabel("test accuracy"); plt.tight_layout(); plt.show()
print(f"\n👉 同一個模型、同一份資料，只是『切法』不同，準確率就從 {min(accs):.0%} 跳到 {max(accs):.0%}。")
print(f"   測試集只有 {len(yte)} 個人，多抽到幾個難判斷的病人，成績就差很多。要小心『一次切分的結果』。")
print("\n👉 請試試看：把上面的 TEST_SIZE 改成 0.5、0.9，再重新執行。測試集變大、訓練集變小，成績會有什麼不同？為什麼？")

**🎓 提醒（2-1 互動）**
- 叫出右側的 **Gemini**，請它「逐行解釋這段程式」，帶學生看懂 `train_test_split`、`test_size`、`random_state`。
- 示範把 `TEST_SIZE` 改成 **0.9**：訓練只剩約 30 人，模型學得少，成績下降。您實測約 70%；我這邊（5 個 seed）約 74–79%，差異來自模型初始化等細節，**以您現場看到的為準**。
- 順便讓學生把 `range(5)` 的種子換掉，看「同一份資料、不同切法」的波動。

### stratify（分層）是什麼？

**stratify**＝切分時，讓**每一份**資料裡「有病／沒病」的比例，都跟全體差不多。
資料少、或疾病罕見時，如果單純隨機切，**運氣不好**就可能出現「測試集裡幾乎沒有病人」。下面的 `stratify=` 就是在做這件事。

In [ ]:
#@title 2-2　切成 訓練／驗證／測試 三份，並把「測試集」鎖起來
TEST_SIZE, VAL_SIZE, SEED = 0.20, 0.20, 42        # 👉 可以改改看

is_yes = df_clean["disease"] == "yes"
trainval_df, test_df = train_test_split(df_clean, test_size=TEST_SIZE, random_state=SEED, stratify=is_yes)
train_df, val_df = train_test_split(trainval_df, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED,
                                    stratify=trainval_df["disease"] == "yes")      # stratify：讓每份的『有病比例』差不多

summary = pd.DataFrame({
    "人數": [len(train_df), len(val_df), len(test_df)],
    "有病比例": [(d["disease"] == "yes").mean() for d in (train_df, val_df, test_df)],
}, index=["訓練集 train", "驗證集 validation", "測試集 test"])
summary["有病比例"] = summary["有病比例"].map("{:.1%}".format)
show(summary, index=True)

test_df.to_csv(os.path.join(WORKDIR, "heart_test_locked.csv"), index=False)
del test_df
print("🔒 測試集已存到 Drive，並從記憶體刪除。單元 4 才會打開它，而且只打開一次。")

Xtr, ytr = to_xy(train_df); Xva, yva = to_xy(val_df)
model, _ = train_mlp(Xtr, ytr)
print(f"\n用訓練集學習，在驗證集上的準確率：{evaluate(model, Xva, yva)['acc']:.1%}")

In [ ]:
#@title 2-3　如果「不 stratify」會怎樣？（故意把切分弄得不平均）
# ① 隨機切 20 次，看測試集裡「有病的比例」會晃多大
rates = {}
for use_strat in (False, True):
    r = []
    for seed in range(20):
        _, te = train_test_split(df_clean, test_size=0.2, random_state=seed, stratify=(is_yes if use_strat else None))
        r.append((te["disease"] == "yes").mean())
    rates[use_strat] = r
show(pd.DataFrame({"切法": ["不 stratify", "stratify"],
                   "測試集有病比例：最低": [f"{min(rates[False]):.1%}", f"{min(rates[True]):.1%}"],
                   "最高": [f"{max(rates[False]):.1%}", f"{max(rates[True]):.1%}"]}))
print(f"全體的有病比例是 {is_yes.mean():.1%}。不 stratify：每次切出來的測試集，有病比例都不一樣。")

# ② 更極端：先依 disease 排序（沒病的在前、有病的在後），再「不洗牌」直接切前 80%／後 20%
srt = df_clean.sort_values("disease").reset_index(drop=True)
tr_bad, te_bad = train_test_split(srt, test_size=0.2, shuffle=False)
tr_ok, te_ok = train_test_split(srt, test_size=0.2, random_state=0, stratify=srt["disease"] == "yes")
rows = []
for name, tr_, te_ in [("❌ 排序後直接切（不洗牌、不 stratify）", tr_bad, te_bad), ("✅ 洗牌並 stratify", tr_ok, te_ok)]:
    Xa, ya = to_xy(tr_); Xb, yb = to_xy(te_)
    model, _ = train_mlp(Xa, ya)
    rows.append({"切法": name, "訓練集有病比例": f"{(tr_['disease'] == 'yes').mean():.0%}",
                 "測試集有病比例": f"{(te_['disease'] == 'yes').mean():.0%}", "測試準確率": f"{evaluate(model, Xb, yb)['acc']:.1%}"})
show(pd.DataFrame(rows))
print("👉 左邊的測試集『全部都是病人』，訓練集的病人卻只有三成：模型沒學到該學的分布，這個準確率也無法反映它在真實世界的表現。")

In [ ]:
#@title 2-4　K 折交叉驗證：資料太少時，讓每一位病人都輪流當一次「驗證」
K = 5
Xtv, ytv = to_xy(trainval_df)
folds = list(StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED).split(Xtv, ytv))

# ① 先看「誰在哪一次當驗證」（還沒開始訓練）
member = np.zeros((K, len(trainval_df)))                 # 0＝這次拿去訓練，1＝這次拿去驗證
for k, (tr, va) in enumerate(folds):
    member[k, va] = 1
from matplotlib.colors import ListedColormap
order = np.argsort(member.argmax(axis=0))
plt.figure(figsize=(12, 3.8))
plt.imshow(member[:, order], aspect="auto", cmap=ListedColormap(["#9ecae1", "#de2d26"]), interpolation="nearest")
plt.yticks(range(K), [f"round {i+1}" for i in range(K)]); plt.xlabel(f"{len(trainval_df)} patients (blue = train, red = validation)")
plt.tight_layout(); plt.show()
print("每一橫列是一次訓練：紅色＝這次拿去『驗證』的病人，藍色＝這次拿去『訓練』的病人。")
print("每位病人剛好當一次驗證、四次訓練；5 次的驗證成績平均起來，比只切一次可靠。")

# ② 再依序訓練、評分
fold_acc = []
for k, (tr, va) in enumerate(folds):
    model, _ = train_mlp(Xtv[tr], ytv[tr], seed=k)
    fold_acc.append(evaluate(model, Xtv[va], ytv[va])["acc"])
    print(f"第 {k+1} 次：訓練 {len(tr)} 人、驗證 {len(va)} 人 → 驗證準確率 {fold_acc[-1]:.1%}")
print(f"\n平均 {np.mean(fold_acc):.1%} ± {np.std(fold_acc):.1%}（± 是 5 次之間的標準差）")
print("👉 報告成績時，請報『平均 ± 標準差』，不要只報某一次切分的最好結果。")

### 其他「輪流當驗證」的方式


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">方法</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">做法</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">訓練幾次</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">適合</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>K 折（K-fold）</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">切成 K 份，輪流 1 份當驗證</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">K 次</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">最常用，K 通常取 5 或 10</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>Leave-one-out（LOO，留一法）</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">每次只留 <b>1 個樣本</b>當驗證，其餘全部訓練</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">N 次（N＝樣本數）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">資料<b>非常少</b>時；但很慢，而且每次驗證只有 1 筆，成績較不穩定</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>Leave-one-patient-out（LOPO，留一病人法）</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">每次留 <b>1 位病人</b>（的<b>所有</b>列）當驗證</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">N 次（N＝病人數）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>一位病人有多筆資料</b>（多次檢查、多張切片）時，必須用這種「以病人分組」的切法，否則會洩漏（見下一節）</td></tr></table>


In [ ]:
#@title 圖｜K 折、LOO、LOPO 的差別
show_fig('splits')

不同的「輪流當驗證」方式 K 折（K＝5） 切成 5 份，輪流 1 份當驗證 訓練 5 次 Leave-one-out 每次只留「1 個樣本」驗證 ⋮ N 個樣本 → 訓練 N 次 Leave-one-patient-out 每次留「1 位病人」（的所有列）驗證 N 位病人 → 訓練 N 次 資料很少時：Leave-one-out 讓訓練資料用得最多，但要訓練 N 次，而且每次驗證只有 1 筆，成績不穩定。 一位病人有多筆資料時：一定要用 Leave-one-patient-out（或 K 折的「以病人分組」版本），否則會洩漏。 紅＝驗證，藍＝訓練；黑色粗線分隔不同病人。

In [ ]:
#@title 2-5　Leave-one-out 要訓練幾次？
from sklearn.model_selection import LeaveOneOut
loo = LeaveOneOut()
n_fits = loo.get_n_splits(Xtv)
t0 = time.time(); train_mlp(Xtv, ytv); one_fit = time.time() - t0
print(f"{len(Xtv)} 位病人 → Leave-one-out 要訓練 {n_fits} 次（K 折只要 5 次）。")
print(f"訓練一次約 {one_fit:.2f} 秒 → 全部約 {n_fits * one_fit / 60:.1f} 分鐘；而且每次的驗證集只有 1 個人。")
tr0, va0 = next(loo.split(Xtv))
print(f"第 1 次：訓練 {len(tr0)} 人、驗證 {len(va0)} 人（就是 trainval_df 的第 {va0[0] + 1} 位病人）。")

RUN_LOO = False        # 👈 想真的跑一遍（約 1 分鐘）就改成 True
if RUN_LOO:
    hits = []
    for k, (tr, va) in enumerate(loo.split(Xtv)):
        model, _ = train_mlp(Xtv[tr], ytv[tr], seed=k)
        hits.append(float(((predict_proba(model, Xtv[va]) >= 0.5) == ytv[va]).all()))
    print(f"Leave-one-out 準確率：{np.mean(hits):.1%}")

### ⚠️ 資料洩漏（data leakage）：模型「偷看了答案」

模型在訓練時，**不小心拿到了「測試時不該知道的資訊」**，成績就會好到不真實。最常見的三種：

In [ ]:
#@title 圖｜洩漏①：病人層級
show_fig('leak_patient')

洩漏 ①　病人層級：同一位病人的多次檢查，分到訓練與測試兩邊 ❌ 以「列」切分 P001 #1 #2 #3 #4 #5 P002 #1 #2 #3 #4 #5 P003 #1 #2 #3 #4 #5 P004 #1 #2 #3 #4 #5 ✅ 以「病人」切分 P001 #1 #2 #3 #4 #5 P002 #1 #2 #3 #4 #5 P003 #1 #2 #3 #4 #5 P004 #1 #2 #3 #4 #5 訓練集 測試集 同一位病人的 5 次檢查長得非常像、答案也一樣。 左邊：模型只要「認得這個人」就能答對測試題 → 成績虛高。

In [ ]:
#@title 圖｜洩漏②：前處理
show_fig('leak_preproc')

洩漏 ②　前處理：先對「全部」資料標準化，再切分 ❌ 洩漏 全部資料 用全部資料 算平均與標準差 標準化 切成 訓練／測試 ✅ 正確 全部資料 先切成 訓練／測試 只用訓練集 算平均與標準差 套用到 訓練與測試 測試集的資訊已經「滲」進平均與標準差 → 模型偷看了考卷 任何「從資料算出來的東西」：平均、標準差、補缺值用的中位數、挑特徵… 都只能用訓練集算，再「套用」到驗證、測試、新病人。

In [ ]:
#@title 圖｜洩漏③：特徵
show_fig('leak_feature')

洩漏 ③　特徵：不小心包含了「答案的線索」 例一：影像上的醫院標記 ICU A 醫院（重症較多） OPD B 醫院（門診） 病人多的醫院，影像角落的字也特別多 模型學會「看角落的字」 而不是「看肺部」→ 換家醫院就失靈 例二：特徵發生在診斷「之後」 症狀出現 診斷 開始用藥 預測的時間點 特徵：用藥紀錄 「正在使用硝化甘油」 早就暗示了心絞痛的診斷 特徵一定要是「預測當下」 就拿得到的資訊

下面先動手做第 ① 種（病人層級）；第 ② 種在單元 3 的標準化之後做；第 ③ 種請記得：**特徵一定要是「預測當下」就拿得到的資訊**。

### ① 病人層級洩漏：動手做做看

想像每位病人做了**好幾次檢查**（或一段 EEG 被切成好幾個片段、一份 MRI 被切成好幾張切片）。資料會長這樣（數值僅為示意）：


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">patient_id</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">第幾次檢查</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">resting_bp</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">chol</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">max_hr</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">disease</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P001</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">1</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">145</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">233</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">150</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">no</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P001</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">2</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">148</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">229</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">153</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">no</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P001</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">3</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">143</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">236</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">148</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">no</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P002</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">1</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">160</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">286</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">108</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">yes</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">P002</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">2</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">157</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">291</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">110</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">yes</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">…</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">…</td></tr></table>


同一位病人的好幾列，數值**非常接近**，答案（`disease`）也**一模一樣**。

- **如果把這些「列」全部打散再隨機切分**：P001 的第 1、3 次檢查可能進了訓練集，第 2 次進了測試集。模型不需要真的學會判斷疾病，只要「認得 P001 長這樣」就能答對測試題。
- **正確做法**：先決定「哪些**病人**」當訓練、哪些當測試，再把每位病人的**所有檢查**一起帶走。

### 這個實驗怎麼做
1. 把我們的 242 位病人，每人複製成 5 列，每列的數值加上約 3% 的隨機誤差（模擬 5 次檢查）→ 共 1210 列。
2. 用兩種切法，各把資料分成 80% 訓練、20% 測試：
   - ❌ **A：以「列」切分**——把 1210 列打散後隨機切
   - ✅ **B：以「病人」切分**——先切病人，再帶走他們全部的列
3. 兩種切法用**同一個模型、同樣的訓練方式**（刻意用比較大的網路：越大的模型越會「背答案」，洩漏的傷害就越大）。
4. 比較**測試集**準確率，各做 4 次取平均。

In [ ]:
#@title 2-6　先看資料長怎樣，以及兩種切法把「同一位病人」的檢查分到哪裡
rng = np.random.default_rng(0)

def make_multi_visit(d, visits=5, jitter=0.03):
    # 假設每位病人檢查了 5 次，每次的數值有一點點測量誤差
    out = []
    for v in range(visits):
        e = d.copy(); e["visit"] = v + 1
        for c in ["age", "resting_bp", "chol", "max_hr", "oldpeak"]:
            e[c] = (e[c] * rng.normal(1, jitter, len(e))).round(1)
        out.append(e)
    return pd.concat(out, ignore_index=True)

multi = make_multi_visit(trainval_df)
Xm, ym = to_xy(multi); groups = multi["patient_id"].values
print(f"{len(trainval_df)} 位病人 × 5 次檢查 = {len(multi)} 列。其中兩位病人的資料：")
two = trainval_df["patient_id"].iloc[:2].tolist()
show(multi[multi["patient_id"].isin(two)].sort_values(["patient_id", "visit"])
     [["patient_id", "visit", "age", "resting_bp", "chol", "max_hr", "disease"]])

# 兩種切法（random_state=0）會把每一列分到「訓練」還是「測試」？
tr_a, te_a = train_test_split(np.arange(len(multi)), test_size=0.2, random_state=0)
tr_b, te_b = next(GroupShuffleSplit(1, test_size=0.2, random_state=0).split(Xm, ym, groups))
side = pd.DataFrame({"patient_id": groups, "visit": multi["visit"], "A": "train", "B": "train"})
side.loc[te_a, "A"] = "test"; side.loc[te_b, "B"] = "test"
pa = side.pivot(index="patient_id", columns="visit", values="A")
pb = side.pivot(index="patient_id", columns="visit", values="B")
split_ids = pa[(pa == "test").any(axis=1) & (pa == "train").any(axis=1)].head(5).index
pa.columns = pb.columns = [f"第{v}次" for v in pa.columns]
print("\n❌ A：以『列』切分——同一位病人的 5 次檢查，被拆到訓練與測試兩邊：")
show(pa.loc[split_ids], index=True)
print("✅ B：以『病人』切分——同樣這幾位病人，5 次檢查全部在同一邊：")
show(pb.loc[split_ids], index=True)

In [ ]:
#@title 2-7　Leave-one-patient-out：每次把「一位病人的全部檢查」留下來驗證
from sklearn.model_selection import LeaveOneGroupOut
logo = LeaveOneGroupOut()
n_splits = logo.get_n_splits(groups=groups)
tr, te = next(logo.split(Xm, ym, groups))
held = multi.iloc[te][["patient_id", "visit"]]
print(f"{len(trainval_df)} 位病人（共 {len(multi)} 列）→ Leave-one-patient-out 要訓練 {n_splits} 次。")
print("第 1 次被留下來驗證的，是這位病人的『全部』檢查：")
show(held)
print(f"訓練集裡還找得到這位病人嗎？　{held['patient_id'].iloc[0] in set(multi.iloc[tr]['patient_id'])}")
print("👉 這樣，模型就不可能靠『認得這個人』來答題。")

In [ ]:
#@title 2-8　洩漏實驗：兩種切法，測試準確率差多少？
def fit_eval(tr, te, seed):
    sc = StandardScaler().fit(Xm[tr])        # 標準化：單元 3 會細講，這裡先用，讓模型學得更好
    model, _ = train_mlp(sc.transform(Xm[tr]), ym[tr], lr=0.01, seed=seed, hidden=(64, 32))
    # 這裡刻意用「大」的網路：越大的模型越會『背答案』，洩漏的傷害就越大
    return evaluate(model, sc.transform(Xm[te]), ym[te])["acc"]

acc_row, acc_pat = [], []
for seed in range(4):
    tr, te = train_test_split(np.arange(len(multi)), test_size=0.2, random_state=seed)              # ❌ A：以「列」切分
    acc_row.append(fit_eval(tr, te, seed))
    tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(Xm, ym, groups))     # ✅ B：以「病人」切分
    acc_pat.append(fit_eval(tr, te, seed))

print(f"❌ A：以『列』隨機切分（同一位病人跨訓練/測試）：測試準確率 {np.mean(acc_row):.1%}")
print(f"✅ B：以『病人』切分（每位病人只在一邊）　　　：測試準確率 {np.mean(acc_pat):.1%}")
plt.figure(figsize=(10.5, 6.8)); plt.bar(["A: split by row\n(leaky)", "B: split by patient\n(correct)"], [np.mean(acc_row), np.mean(acc_pat)],
                                      color=["#de2d26", "#3182bd"]); plt.ylim(0.5, 1); plt.ylabel("test accuracy"); plt.tight_layout(); plt.show()
print("\n👉 好到不真實的成績，要先懷疑『有沒有洩漏』，再開心。")

**🎓 老師備註（單元 2）**
- 2-1 的準確率會明顯跳動（小網路、只做粗略縮放，準確率大約 70–90%）：這是刻意的，單元 3 會示範更好的縮放。
- 2-8 的差距大約是 99％ vs 78％。可以問：「如果這是一篇論文的結果，你會怎麼發現？」→ 看切分單位是病人還是影像／切片。
- 也可提：EEG 的 epoch、MRI 的切片、同一病人的多次住院，都是同樣的結構。

---
## 單元 3｜前處理如何影響訓練

接下來每個實驗都用**同一批資料、同一個小網路**，只改「前處理」這一件事。
為了不被運氣影響，每個實驗都用 5 折交叉驗證重複 3 次（共 15 次訓練），報告**平均 ± 標準差**。
（只用訓練＋驗證資料；測試集還鎖著。）

我們用的網路超小：**輸入 → 3 個感知器 → 1 個感知器，總共 4 個**（約 70 個參數）。資料只有 300 筆，網路大了只會背答案。

### 為什麼要標準化（正規化）？

想像兩個特徵：**體重（約 40–120 kg）**與**血鉀（約 3.5–5.5 mmol/L）**。
數字大的變數，在模型眼中「聲音比較大」。我們不講公式，只講：**把大家拉到同一把尺**。

「聲音大」為什麼會出問題？因為神經網路是靠**梯度**來調整參數的：**輸入越大，梯度越大，參數一次被「推」得越多**。
下面用**一個感知器**（ReLU）親手算一遍：把 loss 傳回去，看參數會被更新多少。

In [ ]:
#@title 圖｜標準化：把大家拉到同一把尺
show_fig('scale')

標準化：把大家拉到「同一把尺」 標準化之前（同一條數線） 0 40 80 120 體重　40–120 kg 血鉀　3.5–5.5 mmol/L（擠在一小點） 體重的數字比血鉀大 20 倍以上 → 在模型眼中，「體重」的聲音大得多 標準化之後（減平均、除以標準差） -2 -1 0 1 2 體重 血鉀 兩個變數都落在差不多的範圍 → 公平競爭

In [ ]:
#@title 圖｜一個感知器：往前算、把 loss 傳回去
show_fig('chain')

一個感知器：往前算（預測），再把 loss 一路「傳回去」（連鎖律） x₁ 體重 x₂ 血鉀 z＝w₁x₁＋w₂x₂＋b ×w₁ ×w₂ ReLU y＝max(z, 0) loss＝(y − t)² t＝正確答案 L ∂L/∂y ＝ 2(y−t) ∂y/∂z ＝ 1（z>0）或 0 ∂z/∂wᵢ ＝ xᵢ 梯度 ∂L/∂wᵢ ＝ 2(y−t) × ∂y/∂z × xᵢ　　更新：wᵢ ← wᵢ − 學習率 × 梯度 輸入 xᵢ 越大，梯度越大，這個權重被「推」得越多

In [ ]:
#@title 3-1　動手算：一個感知器，把 loss 傳回去，看參數被更新多少
# 👈 可以改改看這些數字！
x  = [85, 4.6]       # 輸入：體重 85 kg、血鉀 4.6 mmol/L
w  = [0.1, 0.1]      # 兩個權重（一個對體重、一個對血鉀）
b  = 0.0             # 偏差
t  = 1.0             # 正確答案（希望模型輸出的值）
lr = 0.01            # 學習率：每次更新「走多大一步」

x, w = np.array(x, float), np.array(w, float)
# 【往前算】
z = (w * x).sum() + b                    # 加權總和
y = max(z, 0.0)                          # ReLU：負的變 0
loss = (y - t) ** 2                      # loss：和正確答案差多少（平方）
# 【把 loss 傳回去：連鎖律】
dL_dy = 2 * (y - t)                      # loss 對 y 的微分
dy_dz = 1.0 if z > 0 else 0.0            # ReLU 的微分：z>0 是 1，否則 0
dz_dw = x                                # z 對每個 w 的微分，就是對應的輸入 x
grad_w = dL_dy * dy_dz * dz_dw           # 三個相乘＝loss 對每個 w 的梯度
grad_b = dL_dy * dy_dz

print("【往前算】")
print(f"  z = w₁x₁ + w₂x₂ + b = {w[0]}×{x[0]:g} + {w[1]}×{x[1]:g} + {b} = {z:.3f}")
print(f"  y = ReLU(z) = max({z:.3f}, 0) = {y:.3f}")
print(f"  loss = (y − t)² = ({y:.3f} − {t:g})² = {loss:.3f}")
print("【把 loss 傳回去（連鎖律）】")
print(f"  ∂L/∂y = 2(y − t) = {dL_dy:.3f}")
print(f"  ∂y/∂z = {dy_dz:.0f}　（ReLU：z>0 → 1，否則 0）")
print(f"  ∂z/∂w₁ = x₁ = {x[0]:g}　　∂z/∂w₂ = x₂ = {x[1]:g}")
print(f"  ∂L/∂w₁ = {dL_dy:.3f} × {dy_dz:.0f} × {x[0]:g} = {grad_w[0]:.3f}")
print(f"  ∂L/∂w₂ = {dL_dy:.3f} × {dy_dz:.0f} × {x[1]:g} = {grad_w[1]:.3f}")
print("【更新】新的 w ＝ 舊的 w − 學習率 × 梯度")
print(f"  w₁（體重）：{w[0]} → {w[0] - lr * grad_w[0]:.3f}　（改變 {-lr * grad_w[0]:+.3f}）")
print(f"  w₂（血鉀）：{w[1]} → {w[1] - lr * grad_w[1]:.3f}　（改變 {-lr * grad_w[1]:+.3f}）")

# 用 PyTorch 的自動微分驗證
wt = torch.tensor(w, dtype=torch.float64, requires_grad=True)
(torch.relu((wt * torch.tensor(x)).sum() + b) - t).pow(2).backward()
print("\n✅ PyTorch 自動算出來的梯度也一樣：", np.allclose(wt.grad.numpy(), grad_w))
print(f"👉 體重的權重被推的幅度，是血鉀的 {abs(grad_w[0] / grad_w[1]):.1f} 倍：這就是「輸入大的變數，聲音比較大」。")

In [ ]:
#@title 3-2　同樣的規則，連續更新 5 步：沒標準化 vs 標準化
def run_steps(x, w, b, t, lr, steps=5):
    x, w = np.array(x, float), np.array(w, float)
    rows = []
    for k in range(steps):
        z = (w * x).sum() + b
        y = max(z, 0.0)
        loss = (y - t) ** 2
        g = 2 * (y - t) * (1.0 if z > 0 else 0.0)
        rows.append({"步": k, "z": round(z, 2), "y（ReLU 後）": round(y, 2), "loss": round(loss, 3),
                     "w₁（體重）": round(w[0], 3), "w₂（血鉀）": round(w[1], 3), "梯度 w₁": round(g * x[0], 2) + 0.0})
        w = w - lr * g * x
        b = b - lr * g
    return pd.DataFrame(rows)

# 標準化：假設體重平均 65、標準差 10；血鉀平均 4.2、標準差 0.4 → 85 kg 變成 2.0，4.6 變成 1.0
print("❌ 沒標準化：x ＝ [85, 4.6]，學習率 0.01")
show(run_steps([85, 4.6], [0.1, 0.1], 0.0, 1.0, 0.01))
print("✅ 標準化之後：x ＝ [2.0, 1.0]，同樣的學習率 0.01")
show(run_steps([2.0, 1.0], [0.1, 0.1], 0.0, 1.0, 0.01))
print("👉 沒標準化：第一步 w₁ 就被推到 −13 以下，z 變成很大的負數，ReLU 輸出 0、梯度也變 0 → 參數再也不動（神經元「死掉」）。")
print("   標準化之後：同樣的學習率，loss 穩穩下降。")

In [ ]:
#@title 3-3　標準化：把每個欄位拉到同一把尺上
cols = ["age", "resting_bp", "chol", "max_hr", "oldpeak", "ca"]
print("縮放之前，各欄位的『大小』差很多：")
show(trainval_df[cols].describe().loc[["mean", "std", "min", "max"]].round(1), index=True)
print("膽固醇動輒 200–300，oldpeak 只有 0–6。數值大的欄位會『壓過』數值小的欄位。")
print("（單元 1、2 的 to_xy 其實偷偷做了『粗略版』：每欄除以一個固定的數字，縮到大約 0–1。現在來比較三種做法。）\n")

def loss_curves(scale):
    prep = make_prep("onehot", scale)
    Xtr = prep.fit_transform(add_male(train_df)).astype("float32")       # 只用「訓練集」算出縮放需要的數字
    Xva = prep.transform(add_male(val_df)).astype("float32")             # 驗證集只能「套用」，不能參與計算
    ytr = (train_df["disease"] == "yes").values.astype("float32"); yva = (val_df["disease"] == "yes").values.astype("float32")
    return train_mlp(Xtr, ytr, X_val=Xva, y_val=yva)[1]

curves = {"no scaling": loss_curves("none"), "rough (divide by fixed number)": loss_curves("rough"), "standardized": loss_curves("standard")}
fig, ax = plt.subplots(1, 2, figsize=(15.0, 4.5), sharey=True)
for a_, key, title in zip(ax, ["train", "val"], ["training loss", "validation loss"]):
    for name, h in curves.items():
        a_.plot(h[key], label=name)
    a_.set_title(title); a_.set_xlabel("epoch"); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

RESULTS.clear()
print("五折交叉驗證 × 3 次，三種做法的成績：")
_ = cv_eval(trainval_df, "S0. no scaling", encode="onehot", scale="none")
_ = cv_eval(trainval_df, "S1. rough scaling", encode="onehot", scale="rough")
res_std = cv_eval(trainval_df, "S2. standardize", encode="onehot", scale="standard")
print("\n👉 不縮放：成績明顯較低，而且很不穩定（± 很大）；縮放之後又好又穩。標準化＝減平均、除以標準差，讓每欄平均 0、標準差 1。")

**標準化的重點**：`(數值 − 訓練集的平均) ÷ 訓練集的標準差`。注意這裡的「平均」和「標準差」是**從訓練集算出來**，之後套用到驗證集、測試集、新病人身上。

**🎓 老師備註**：預期 S0 約 70％（± 約 14，很不穩定）、S1 約 82％、S2 約 83％。看 loss 曲線：S0 下降慢又抖。S1 vs S2 的差距不大，可以說明「粗略縮放也能用，標準化是更標準、更穩的版本」。

In [ ]:
#@title 3-4　類別資料：整數編碼 vs One-hot（之後的實驗都使用標準化）
print("原始表格：類別是文字")
show(trainval_df[["patient_id", "chest_pain", "thal"]].head(4))

print("\nOne-hot：每個類別變成一欄 0/1")
oh = pd.get_dummies(trainval_df[["chest_pain"]]).astype(int)
show(pd.concat([trainval_df[["patient_id", "chest_pain"]], oh], axis=1).head(4))

print("\n整數編碼會是 typical=0, nontypical=1, nonanginal=2, asymptomatic=3。")
print("模型會以為 asymptomatic(3) 是 typical(0) 的『三倍』、nonanginal 在『中間』，但這個順序是我們隨便排的。")
print("\n各類型實際的有病比例：")
show((trainval_df.groupby("chest_pain")["disease"].apply(lambda s: (s == "yes").mean())
      .round(2).to_frame("有病比例")), index=True)

print("\n三種做法的比較：")
_ = cv_eval(trainval_df, "A. numbers only (no text cols)", encode="none")
_ = cv_eval(trainval_df, "B. integer codes", encode="int")
_ = cv_eval(trainval_df, "C. one-hot", encode="onehot")

**怎麼讀這個結果？** 加進胸痛類型與 thal 之後，成績有沒有變好（A → B/C）？One-hot 比整數編碼好多少？看看差距有沒有大過 ± 的範圍。
資料很少時，**小的差異可能只是運氣**，這就是為什麼我們要看平均 ± 標準差。

**🎓 老師備註**：A → B/C 有進步；B vs C 常在雜訊範圍內。可以誠實地說：「在這個小網路與小資料上，one-hot 的好處不大；它真正重要的場景是類別很多（例如 ICD 碼）或線性模型。但**整數編碼會引入不存在的順序，是概念上的錯誤**。」

### ⚠️ 洩漏（一）：標準化要用「誰」的平均與標準差？

標準化要先算出「平均」和「標準差」。問題是：這兩個數字，是用**哪些人**算的？


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left"></th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">❌ 洩漏的做法</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">✅ 正確的做法</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">第 1 步</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">用<b>全部</b> 242 人算平均與標準差</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">先分出「訓練折」與「驗證折」</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">第 2 步</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">再把 242 人分成訓練／驗證</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">只用「訓練折」的人算平均與標準差</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">結果</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">驗證的人<b>早就參與了計算</b>，他們的資訊「滲」進了訓練</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">驗證的人只是被「套用」這組數字，沒有參與計算</td></tr></table>


（這裡「驗證折」扮演的，就是「之後才會來考試的病人」。）

In [ ]:
#@title 3-5　⚠️ 洩漏：「先標準化全部資料、再切分」 vs 「只用訓練集的平均與標準差」
d = add_male(trainval_df).reset_index(drop=True)
y = (d["disease"] == "yes").astype(float).values
X_leaky = make_prep("onehot", True).fit_transform(d).astype("float32")       # ❌ 平均/標準差用了全部資料，包含之後要拿來『考試』的人
rows = []
for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0).split(d, y)):
    model, _ = train_mlp(X_leaky[tr], y[tr], seed=k)
    rows.append(evaluate(model, X_leaky[va], y[va]))
r = pd.DataFrame(rows)
proper = res_std
print(f"❌ 先標準化再切分：accuracy {r['acc'].mean():.3f}   AUC {r['auc'].mean():.3f}")
print(f"✅ 只用訓練集　　：accuracy {proper['acc']:.3f}   AUC {proper['auc']:.3f}")
print("\n這個資料集上兩者幾乎一樣：因為只是一點點平均值與標準差的資訊『滲』過去。")
print("但原則不能破：任何『從資料算出來的東西』（平均、標準差、補缺值用的中位數、挑特徵）都只能用訓練資料算。")

### ⚠️ 洩漏（二）：如果被偷看的是「答案」呢？

**設定**：242 位病人，每人有 **300 個純雜訊特徵**（隨機數字，沒有任何意義），標籤（有病／沒病）也是**隨機決定**的。
所以不管用什麼方法，理論上都只能猜對 **50%**。

**任務**：從 300 個特徵裡，挑出和標籤最相關的 10 個，再用這 10 個特徵訓練模型。


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left"></th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">❌ 洩漏的做法</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">✅ 正確的做法</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">怎麼挑特徵</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">用<b>全部</b> 242 人（包含之後要驗證的人）的答案來挑</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">只用「訓練折」的人的答案來挑</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">怎麼評分</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">在驗證折的人身上</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">在驗證折的人身上</td></tr></table>


（這裡**沒有動用鎖住的測試集**；「驗證折」扮演「之後才來考試的人」。）

**為什麼洩漏會變準？** 300 個雜訊裡，總會有幾個「碰巧」和答案相關。用全部人的答案去挑，等於偷看考卷，挑出「碰巧對」的特徵；之後驗證時，這些特徵當然「碰巧」又對。

In [ ]:
#@title 3-6　（加碼，時間夠再跑）洩漏有多可怕？用「完全隨機」的資料看看
rng = np.random.default_rng(0)
n = len(trainval_df)
X_noise = rng.normal(size=(n, 300)).astype("float32")      # 300 個純雜訊的「基因」
y_noise = rng.integers(0, 2, n).astype("float32")          # 隨機決定有病或沒病：根本不可能預測

def top_k(X, y, k=10):
    Xc = (X - X.mean(0)) / X.std(0); yc = (y - y.mean()) / y.std()
    return np.argsort(-np.abs((Xc * yc[:, None]).mean(0)))[:k]

leaky, proper_ = [], []
for k, (tr, va) in enumerate(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0).split(X_noise, y_noise)):
    sel = top_k(X_noise, y_noise)                      # ❌ 用「全部」資料的答案來挑特徵
    m, _ = train_mlp(X_noise[tr][:, sel], y_noise[tr], seed=k, lr=0.01); leaky.append(evaluate(m, X_noise[va][:, sel], y_noise[va])["acc"])
    sel = top_k(X_noise[tr], y_noise[tr])              # ✅ 只用訓練折挑特徵
    m, _ = train_mlp(X_noise[tr][:, sel], y_noise[tr], seed=k, lr=0.01); proper_.append(evaluate(m, X_noise[va][:, sel], y_noise[va])["acc"])
print(f"❌ 先用全部資料挑特徵：準確率 {np.mean(leaky):.1%}")
print(f"✅ 只用訓練資料挑特徵：準確率 {np.mean(proper_):.1%}　（隨機資料，理論上就是 50％）")
print("\n👉 資料裡完全沒有訊號，洩漏卻讓模型看起來『有預測力』。基因體學、影像組學（特徵多、人數少）的論文最容易中招。")

In [ ]:
#@title 3-7　資料擴增：表格資料加雜訊
_ = cv_eval(trainval_df, "E. + noise augmentation (x3)", encode="onehot", scale="standard", augment=noise_augment(3, 0.1))
print("\n👉 在這個小資料集上，擴增幾乎沒有幫助（差異在 ± 範圍內）。擴增不是萬靈丹，一樣要用驗證的方式檢查有沒有用。")
print("   而且：只能擴增『訓練集』。如果驗證集也擴增，同一個病人的變形版本就會跨過兩邊，又是洩漏。")

In [ ]:
#@title 3-8　影像資料的擴增（純示範，不訓練）：哪些變形在醫學上合理？
try:
    from skimage.data import shepp_logan_phantom
    from skimage.transform import resize, rotate
    head = resize(shepp_logan_phantom(), (200, 200))               # 標準的「頭部切面」測試影像（模擬，非真實病人）
except Exception:
    yy0, xx0 = np.mgrid[:200, :200]; head = (((xx0 - 100) / 80) ** 2 + ((yy0 - 100) / 95) ** 2 < 1) * 0.6
    rotate = lambda im, a: im
yy, xx = np.mgrid[:200, :200]
img = head.copy(); img[((xx - 60) ** 2 + (yy - 95) ** 2) < 14 ** 2] = 1.0       # 在影像「左側」放一個假的病灶
panels = [("original", img), ("flip left-right", img[:, ::-1]), ("rotate 10 deg", rotate(img, 10)),
          ("brightness x1.3", np.clip(img * 1.3, 0, 1)),
          ("gaussian noise", np.clip(img + np.random.default_rng(0).normal(0, 0.08, img.shape), 0, 1))]
fig, ax = plt.subplots(1, 5, figsize=(13, 3))
for a, (t, im) in zip(ax, panels):
    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()
print("左右翻轉之後，病灶跑到另一側。如果你的任務跟『哪一側』有關（例如左/右側中風），這種擴增會直接把標籤弄錯。")
print("旋轉一點點、調亮度、加雜訊，通常合理（模擬病人擺位、不同掃描儀）。什麼擴增合理，要由懂臨床的人決定。")

In [ ]:
#@title 圖｜類別不平衡
show_fig('imbalance')

類別不平衡：罕見疾病只占 1% 100 位病人裡，只有 1 位有病（紅點） 模型只會一招： 全部猜「沒病」 accuracy＝99％ recall＝0％ （有病的那一位， 一個都沒抓到） 怎麼辦？ 重抽樣 過採樣：複製少數類別 欠採樣：減少多數類別 調整權重 class weight：漏掉 少數類別，罰得更重 選對指標 recall、precision、AUC 別只看 accuracy

### 先認識幾個詞：怎麼評量「抓病人」的能力？

假設 100 位病人，其中 15 位有病。模型的預測結果可以分成四格：


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left"></th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">實際有病</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">實際沒病</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>預測有病</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">TP 真陽性（抓到了）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">FP 偽陽性（誤報）</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>預測沒病</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">FN 偽陰性（<b>漏診</b>）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">TN 真陰性</td></tr></table>


- **accuracy（準確率）**＝(TP＋TN) ÷ 全部人：全部猜對幾成
- **recall（召回率／敏感度）**＝TP ÷ (TP＋FN)：**有病的人，抓到幾成**
- **precision（精確率）**＝TP ÷ (TP＋FP)：**被預測有病的人，真的有病幾成**
- **AUC**：不看門檻，模型把「有病的人」排在「沒病的人」前面的能力（0.5＝亂猜，1＝完美）

三種訓練方式：


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">名稱</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">做法</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>plain</b>（一般訓練）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">什麼都不特別處理。有病的人太少，模型容易偏向猜「沒病」</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>class weight</b>（類別權重）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">訓練時，漏掉一個「有病」的人，損失乘上約（沒病人數 ÷ 有病人數）倍，逼模型重視少數</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>oversample</b>（過採樣）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">把「有病」的人<b>重複抽樣複製</b>，補到跟沒病的一樣多。只對<b>訓練集</b>做，驗證時不做</td></tr></table>


In [ ]:
#@title 3-9　資料不平衡：罕見疾病只占一小部分
pos = trainval_df[trainval_df["disease"] == "yes"]; neg = trainval_df[trainval_df["disease"] == "no"]
n_pos = int(round(0.15 * len(neg) / 0.85))
imb_df = pd.concat([neg, pos.sample(n_pos, random_state=1)]).sample(frac=1, random_state=1).reset_index(drop=True)
yi = (imb_df["disease"] == "yes").astype(float).values
print(f"把有病的人抽樣到約 15％：共 {len(imb_df)} 人，其中有病 {int(yi.sum())} 人（{yi.mean():.0%}）")

def oof_proba(strategy, repeats=3):
    """交叉驗證，把每一位病人『沒被訓練到的那一折』的預測機率收集起來。"""
    P = np.zeros(len(imb_df))
    for rep in range(repeats):
        for j, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=rep).split(imb_df, yi)):
            prep = make_prep("onehot", True)
            Xtr = prep.fit_transform(add_male(imb_df.iloc[tr])).astype("float32"); Xva = prep.transform(add_male(imb_df.iloc[va])).astype("float32")
            ytr, pw = yi[tr], None
            if strategy == "class weight":                       # 讓「錯過一個病人」的懲罰變重
                pw = (ytr == 0).sum() / (ytr == 1).sum()
            if strategy == "oversample":                         # 把少數類別（有病）重複抽樣補到跟多數一樣多
                idx = np.where(ytr == 1)[0]
                extra = np.random.default_rng(j).choice(idx, (ytr == 0).sum() - len(idx), replace=True)
                Xtr, ytr = np.vstack([Xtr, Xtr[extra]]), np.r_[ytr, ytr[extra]]
            model, _ = train_mlp(Xtr, ytr, seed=j, pos_weight=pw)
            P[va] += predict_proba(model, Xva) / repeats
    return P

from sklearn.metrics import confusion_matrix
n_pos_all, n_neg_all = int(yi.sum()), int((1 - yi).sum())
rows = [{"strategy": "always predict 'no disease'", "TP": 0, "FN": n_pos_all, "FP": 0, "TN": n_neg_all,
         "accuracy": 1 - yi.mean(), "recall": 0.0, "precision": float("nan"), "AUC": 0.5}]
for s in ["plain", "class weight", "oversample"]:
    P = oof_proba(s); pred = (P >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yi, pred).ravel()
    rows.append({"strategy": s, "TP": tp, "FN": fn, "FP": fp, "TN": tn, "accuracy": accuracy_score(yi, pred),
                 "recall": recall_score(yi, pred), "precision": precision_score(yi, pred, zero_division=0), "AUC": roc_auc_score(yi, P)})
show(pd.DataFrame(rows).round(3))
print("👉 『全部猜沒病』的準確率就有 85％，recall（有病的人抓到幾成）卻是 0。罕見疾病不能只看 accuracy。")
print("   class weight、過採樣可以多抓到一些病人（recall 上升），代價通常是 precision 下降（誤報變多）。要選哪個，取決於漏診與誤報哪個代價更大。")

In [ ]:
#@title 3-10　總結：每一種前處理，到底值多少？
res = pd.DataFrame(RESULTS)
show(res.round(3))
plt.figure(figsize=(10.5, 4.5)); plt.barh(res["label"][::-1], res["acc"][::-1], xerr=res["acc_sd"][::-1], color="#3182bd")
plt.xlim(0.5, 1); plt.xlabel("cross-validated accuracy (mean ± sd)"); plt.tight_layout(); plt.show()
print("標準化的影響最大；加入類別欄位、one-hot 次之；擴增在這裡沒有顯著差異。")
print("前處理沒有『一定有用』的招式，每一招都要用乾淨的驗證方式檢查。")

---
## 單元 4｜推論：新病人進來了

**訓練（training）** 是在學習；**推論（inference）** 是真的拿來用。
關鍵原則：**推論時對資料做的前處理，必須跟訓練時一模一樣。**
這表示：所有「從訓練資料算出來的東西」（標準化的平均與標準差、one-hot 的欄位順序、補缺值用的中位數）都要**一起存起來**，跟模型成套使用。

In [ ]:
#@title 4-1　定案：看看「前處理」到底記住了哪些數字，再訓練並存起來
import joblib
d_tv = add_male(trainval_df)

# 補缺值用的數字：只用「訓練＋驗證」資料算
fill = {**{c: trainval_df[c].median() for c in ALL_NUMERIC},
        **{c: trainval_df[c].mode()[0] for c in ["sex", "chest_pain", "thal"]}}

# 建立前處理，並且只用「訓練＋驗證」資料 fit（＝讓它記住這些資料的平均、標準差、類別順序）
final_prep = make_prep("onehot", "standard")
X_final = final_prep.fit_transform(d_tv).astype("float32")
y_final = (trainval_df["disease"] == "yes").astype("float32").values

scaler = final_prep.named_transformers_["num"]
print("① 標準化記住的數字：每個連續欄位，在訓練資料裡的平均與標準差")
stats = pd.DataFrame({"欄位": PREP_NUM, "mean 平均": scaler.mean_, "std 標準差": scaler.scale_,
                      "median 中位數（補缺值用）": [fill[c] for c in PREP_NUM]})
show(stats.round(2))
ex = trainval_df["chol"].iloc[0]
print(f"   例：某位病人的膽固醇 {ex:.0f} → (數值 − 平均) ÷ 標準差 = ({ex:.0f} − {scaler.mean_[2]:.1f}) ÷ {scaler.scale_[2]:.1f} = {(ex - scaler.mean_[2]) / scaler.scale_[2]:.2f}")

print("\n② One-hot 記住的：每個類別欄位有哪些類別、順序是什麼（順序錯了，模型就看錯欄位）")
oh = final_prep.named_transformers_["onehot"]
nice = lambda c: [x if isinstance(x, str) else int(x) for x in c]          # 把 0.0 顯示成 0
show(pd.DataFrame({"欄位": PREP_CAT, "類別（依序展開成 0/1 欄）": [nice(c) for c in oh.categories_]}))

print("\n③ 補缺值用的數字（缺值不能用『新資料自己的中位數』，要用訓練時算好的）：")
show(pd.DataFrame({"欄位": list(fill), "補上的值": list(fill.values())}).round(2))

print(f"\n④ 前處理後，模型看到的 {X_final.shape[1]} 個欄位（依序）：")
names = [n.split("__", 1)[1] for n in final_prep.get_feature_names_out()]
groups, i = [], 0
groups.append((f"標準化後的連續欄位（{len(PREP_NUM)}）", PREP_NUM)); i += len(PREP_NUM)
groups.append((f"本來就是 0/1（{len(PREP_BIN)}）", PREP_BIN)); i += len(PREP_BIN)
for col, cats in zip(PREP_CAT, oh.categories_):
    groups.append((f"one-hot：{col}（{len(cats)}）", names[i:i + len(cats)])); i += len(cats)
show(pd.DataFrame({"來源": [g[0] for g in groups], "展開後的欄位（依序）": [", ".join(g[1]) for g in groups]}))

final_model, _ = train_mlp(X_final, y_final, seed=0)
ART = os.path.join(WORKDIR, "artifacts"); os.makedirs(ART, exist_ok=True)
joblib.dump({"prep": final_prep, "fill": fill}, os.path.join(ART, "preprocessor.joblib"))
torch.save(final_model.state_dict(), os.path.join(ART, "model.pt"))
print("\n✅ 已存到 Drive：", sorted(os.listdir(ART)))

In [ ]:
#@title 4-1b　打開剛存的檔案看看：preprocessor.joblib 與 model.pt 裡面是什麼？
bundle = joblib.load(os.path.join(ART, "preprocessor.joblib"))
print("preprocessor.joblib 是一個『包裹』，裡面有：", list(bundle.keys()))
print(f"  • prep：{type(bundle['prep']).__name__}　→ 內含上面的 平均／標準差（①）與 類別順序（②）")
print(f"  • fill：{type(bundle['fill']).__name__}　→ 補缺值用的 {len(bundle['fill'])} 個數字（③）")
print("  → 推論時，只要把新病人丟給 prep.transform(...)，就會自動套用『訓練時』記住的一切。")

state = torch.load(os.path.join(ART, "model.pt"))
show(pd.DataFrame({"層": list(state.keys()), "形狀": [tuple(v.shape) for v in state.values()],
                   "參數個數": [v.numel() for v in state.values()]}))
print(f"model.pt 是模型學到的『權重』：總共只有 {sum(v.numel() for v in state.values())} 個數字（這就是整個 AI）。")

In [ ]:
#@title 4-2　🔓 打開鎖住的測試集（第一步：讀進來）
test_locked = pd.read_csv(os.path.join(WORKDIR, "heart_test_locked.csv"))
test_locked[ALL_NUMERIC] = test_locked[ALL_NUMERIC].astype(float)
print(f"鎖了很久的測試集：{len(test_locked)} 位病人。單元 2 鎖起來之後，沒有人看過它，也沒拿它調整過任何東西。")
print("它是「已經清理好」的資料（文字類別、原始數值，還沒標準化／one-hot）：")
show(test_locked.head(8))

In [ ]:
#@title 4-3　第二步：套用「訓練時記住的」前處理
X_test = final_prep.transform(add_male(test_locked)).astype("float32")
y_test = (test_locked["disease"] == "yes").astype("float32").values
X_test_df = pd.DataFrame(X_test, columns=names)
print("處理完的測試集（連續欄位已標準化、類別已展開成 0/1）：")
show(X_test_df.head(8).round(2))
i = 0
print(f"\n對照第一位病人：膽固醇 {test_locked['chol'].iloc[i]:.0f} → "
      f"({test_locked['chol'].iloc[i]:.0f} − {scaler.mean_[2]:.1f}) ÷ {scaler.scale_[2]:.1f} = {X_test_df['chol'].iloc[i]:.2f}　（用的是『訓練資料』的平均與標準差）")
print(f"胸痛類型 {test_locked['chest_pain'].iloc[i]} → "
      + ", ".join(f"{c}={X_test_df[c].iloc[i]:.0f}" for c in names if c.startswith("chest_pain")))

In [ ]:
#@title 4-4　第三步：模型的推論結果
p = predict_proba(final_model, X_test)
result = pd.DataFrame({"patient_id": test_locked["patient_id"], "預測有病的機率": p.round(3),
                       "預測": np.where(p >= 0.5, "yes", "no"), "實際": test_locked["disease"]})
result["對不對"] = np.where(result["預測"] == result["實際"], "✅", "❌")
print("前 12 位病人的推論結果：")
show(result.head(12))

res = evaluate(final_model, X_test, y_test)
tn = int(((p < 0.5) & (y_test == 0)).sum()); fp = int(((p >= 0.5) & (y_test == 0)).sum())
fn = int(((p < 0.5) & (y_test == 1)).sum()); tp = int(((p >= 0.5) & (y_test == 1)).sum())
print(f"\n全部 {len(y_test)} 位：TP {tp}　FN {fn}　FP {fp}　TN {tn}")
print(f"accuracy {res['acc']:.1%}　recall {res['recall']:.1%}　precision {res['precision']:.1%}　AUC {res['auc']:.3f}")
print("\n👉 這才是對『新病人』的誠實成績。如果現在回頭調整模型再重測，測試集就變成另一份驗證集，失去意義。")

In [ ]:
#@title 4-5　一位新病人走進診間：原始表單 → 同樣的前處理 → 預測
new_patient = {            # 就像從不同系統匯出來的「原始表單」，有各種不整齊的寫法
    "patient_id": "NEW001", "age": "55", "sex": "Male", "chest_pain": "non-anginal",
    "resting_bp": "135 mmHg", "chol": "6.4", "fbs": "0", "rest_ecg": "0", "max_hr": "140",
    "exang": "0", "oldpeak": "1.2", "slope": "2", "ca": "1", "thal": "reversable",
}
print("原始輸入：", {k: v for k, v in new_patient.items() if k in ["sex", "chest_pain", "resting_bp", "chol", "thal"]}, "…")

def predict_risk(raw, prep=final_prep, model=final_model, fill=fill, tweak=None):
    d = auto_clean(pd.DataFrame([raw]), fill=fill)       # 同一套清理規則（含：mmol/L→mg/dL、寫法統一、補值用『訓練時』的數字）
    if tweak: tweak(d)
    return float(predict_proba(model, prep.transform(add_male(d)).astype("float32"))[0])

risk = predict_risk(new_patient)
cleaned = auto_clean(pd.DataFrame([new_patient]), fill=fill)
print(f"清理後：sex={cleaned.sex[0]}, resting_bp={cleaned.resting_bp[0]:.0f}, chol={cleaned.chol[0]:.0f} mg/dL, thal={cleaned.thal[0]}")
print(f"\n✅ 預測有冠心病的機率：{risk:.1%}")

In [ ]:
#@title 4-6　故意做錯：前處理跟訓練時不一致，會發生什麼事？
def no_unit_fix(d): d.loc[0, "chol"] = float(new_patient["chol"])   # 忘了把 mmol/L 換成 mg/dL（直接用 6.4）

# 錯誤 A：跳過標準化（直接把原始數字餵進去）
X_ok = final_prep.transform(add_male(cleaned)).astype("float32")
raw_vals = cleaned[PREP_NUM].values.astype("float32")
X_noscale = X_ok.copy(); X_noscale[:, :len(PREP_NUM)] = raw_vals
risk_noscale = float(predict_proba(final_model, X_noscale)[0])

# 錯誤 B：忘了單位換算
risk_unit = predict_risk(new_patient, tweak=no_unit_fix)

# 錯誤 C：在新資料上「重新」算平均與標準差（只有一個人，標準差是 0）
sc_wrong = StandardScaler().fit(cleaned[PREP_NUM])
X_resc = X_ok.copy(); X_resc[:, :len(PREP_NUM)] = sc_wrong.transform(cleaned[PREP_NUM])
risk_resc = float(predict_proba(final_model, X_resc)[0])

table = pd.DataFrame({"做法": ["✅ 正確：套用訓練時存下來的前處理", "❌ A. 跳過標準化", "❌ B. 忘了膽固醇單位換算", "❌ C. 在新資料上重新標準化"],
                      "預測機率": [risk, risk_noscale, risk_unit, risk_resc]})
table["預測機率"] = table["預測機率"].map("{:.1%}".format)
show(table)

# 錯誤 D：現場用 get_dummies 做 one-hot（新病人只有一位，欄位數、順序都跟訓練時不一樣）
try:
    X_bad = pd.get_dummies(cleaned[["chest_pain", "thal"]]).astype("float32").values
    print(f"\n❌ D. 現場 get_dummies：只得到 {X_bad.shape[1]} 欄；模型需要 {X_final.shape[1]} 欄。")
    predict_proba(final_model, X_bad)
except Exception as e:
    print("   模型直接報錯：", str(e)[:90], "…")
print("\n👉 最危險的是 A、B、C：程式不會報錯，只是預測默默變得不可靠；A 甚至會讓『高風險病人』看起來像『低風險』。")

---
## 單元 4 補充｜去識別化與隱私

我們今天用的是**公開、已去識別**的資料集。但你們以後做研究，拿到的是**院內病人的資料**。
下面先看：研究資料裡，**哪些東西可以認出病人**？

In [ ]:
#@title 圖｜研究資料裡藏著哪些可以認出病人的東西
show_fig('privacy')

研究資料裡，藏著哪些「可以認出病人」的東西？ 🧑 影像：臉部 3D 頭部 MRI／CT 可以重建出臉， 再用人臉辨識比對到真人 對策：去臉（defacing）、 只提供去除顱骨的腦組織 🗂 DICOM 標頭 姓名、病歷號、生日 檢查日期、醫院、醫師 機器序號、各種 UID 對策：刪除／取代／日期位移 UID 重新產生；小心影像內嵌文字 📝 病歷文字 姓名、身分證字號、電話 地址、日期、病歷號 「王大明的女兒」 對策：規則＋自然語言處理， 再人工抽查 去識別化 ≠ 絕對安全 還有「間接識別」：罕見疾病 ＋ 精確日期 ＋ 居住地區，湊在一起也可能認出人。 所以研究要經過 IRB，資料要依法處理，不要自己帶走、不要放到公開的 AI 服務。

### ① 影像：臉部可以被重建
頭部 **MRI／CT** 是 3D 的，把表面重建出來就是一張臉，已有研究證明可以用人臉辨識軟體比對到真人（例如 Schwarz 等人，*NEJM* 2019）。
所以影像研究常要做「**去臉（defacing）**」，或只提供去除顱骨與臉部的腦組織。

### ② DICOM 標頭：影像檔「自己帶著」病人資料
醫學影像（DICOM）檔案除了像素，還有一大段**標頭（header）**。下面是一個**虛構**的範例：

In [ ]:
#@title 4-7　DICOM 標頭長什麼樣子？（範例；病人與資料皆為虛構）
header = [
    # (tag, 名稱, 原始值, 風險, 處理方式, 去識別化後)
    ("(0010,0010)", "PatientName", "CHEN^DAWEN", "直接識別", "取代", "ANON-0042"),
    ("(0010,0020)", "PatientID（病歷號）", "A123456789", "直接識別", "取代（對照表另外保管）", "ANON-0042"),
    ("(0010,0030)", "PatientBirthDate", "19680312", "直接識別", "只留年份", "1968"),
    ("(0010,1040)", "PatientAddress", "台北市中正區某某路 1 號", "直接識別", "刪除", "（空）"),
    ("(0010,2154)", "PatientTelephoneNumbers", "0912-345-678", "直接識別", "刪除", "（空）"),
    ("(0008,0050)", "AccessionNumber", "H2025110400123", "直接識別", "刪除", "（空）"),
    ("(0008,0090)", "ReferringPhysicianName", "WANG^XIAOMING", "直接識別", "刪除", "（空）"),
    ("(0008,1070)", "OperatorsName", "LI^TECH", "直接識別", "刪除", "（空）"),
    ("(0010,0040)", "PatientSex", "M", "間接識別", "保留（研究需要）", "M"),
    ("(0010,1010)", "PatientAge", "057Y", "間接識別", "保留或分組", "050-059Y"),
    ("(0008,0020)", "StudyDate", "20251104", "間接識別", "日期位移（同一病人位移相同天數）", "20250918"),
    ("(0008,0030)", "StudyTime", "093015", "間接識別", "刪除", "（空）"),
    ("(0008,0080)", "InstitutionName", "Example General Hospital", "間接識別", "刪除", "（空）"),
    ("(0018,1000)", "DeviceSerialNumber", "SN-483920", "間接識別", "刪除", "（空）"),
    ("(0008,1030)", "StudyDescription", "MRI BRAIN, r/o stroke", "間接識別", "保留，但要檢查有沒有夾帶姓名", "MRI BRAIN, r/o stroke"),
    ("(0020,000D)", "StudyInstanceUID", "1.2.840.113619.2.55.3.60468…", "間接識別", "重新產生（對照表另外保管）", "2.25.3175…（新的）"),
    ("(0008,0060)", "Modality", "MR", "不需處理", "保留", "MR"),
    ("(0018,0050)", "SliceThickness", "1.0", "不需處理", "保留", "1.0"),
    ("(0028,0030)", "PixelSpacing", "0.5\\0.5", "不需處理", "保留", "0.5\\0.5"),
]
dicom = pd.DataFrame(header, columns=["Tag", "名稱", "原始值", "風險", "處理方式", "去識別化後"])
print("19 個標頭欄位（節錄）。風險：直接識別 ＝ 一看就知道是誰；間接識別 ＝ 湊在一起可能認出人。")
show(dicom)
print("⚠️ 還有一個陷阱：像素本身可能「燒」進了文字（例如超音波影像上的病人姓名），只改標頭不夠，要另外檢查影像內容。")
print("DICOM 標準有制式的去識別化設定檔（PS3.15 附錄 E：Basic Application Level Confidentiality Profile），可以當作清單參考。")

### ③ 病歷文字：姓名、日期、號碼

In [ ]:
#@title 4-8　病歷文字的去識別化：用規則把姓名、日期、號碼拿掉（範例；內容為虛構）
import re
note = ("2025/11/04 門診。病人陳大文（病歷號 12345678，身分證 A123456789，手機 0912-345-678），57 歲男性，"
        "住台北市中正區某某路1號。主訴左側肢體無力 2 小時，由女兒陳小美陪同就醫。2025-11-04 09:30 頭部 MRI 顯示右側 MCA 梗塞。"
        "預計 2025年11月18日 回診，由王小明醫師追蹤。")
names_in_note = ["陳大文", "陳小美", "王小明"]            # 示範用：真實情況要靠名單或自然語言處理（NER）找人名
rules = [(r"[A-Z][12]\d{8}", "[身分證]"), (r"09\d{2}-?\d{3}-?\d{3}", "[電話]"), (r"病歷號\s*\d{6,8}", "病歷號 [病歷號]"),
         (r"\d{4}[/-]\d{1,2}[/-]\d{1,2}(\s*\d{1,2}:\d{2})?", "[日期]"), (r"\d{4}年\d{1,2}月\d{1,2}日", "[日期]"),
         (r"(台北|新北|桃園|台中|台南|高雄)市[^，。\s]*\d+號", "[地址]")]
clean_note, hits = note, {}
for pat, tag in rules:
    clean_note, n = re.subn(pat, tag, clean_note)
    hits[tag] = hits.get(tag, 0) + n
for nm in names_in_note:
    clean_note = clean_note.replace(nm, "[姓名]"); hits["[姓名]"] = hits.get("[姓名]", 0) + note.count(nm)
print("【原文】\n" + note + "\n")
print("【去識別化之後】\n" + clean_note + "\n")
show(pd.DataFrame({"取代成": list(hits), "次數": list(hits.values())}))
print("規則做不到的事：沒列在名單上的人名、「某某的女兒」這類關係、手寫或口語的日期寫法…")
print("更難的是「間接識別」：57 歲 ＋ 左側無力 ＋ 某區居民，湊在一起，罕見的話還是可能認出人。所以去識別化之後仍然要人工抽查。")

### 台灣的相關規範
> 以下是教學用的**重點整理**（資料查詢日期：2026-11-04），**不是法律意見**。實際做研究時，請以全國法規資料庫的現行條文，以及你所屬醫院 **IRB** 的規定為準。


<table style="font-size:17px;border-collapse:collapse;margin:8px 0;max-width:100%"><tr><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">規範</th><th style="border:1px solid #888;padding:6px 12px;background:#e8eaf6;color:#000;text-align:left">與我們的研究資料有關的重點</th></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>個人資料保護法 第 6 條</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>病歷、醫療、基因、性生活、健康檢查、犯罪前科</b>屬「特種個資」，原則上<b>不得蒐集、處理或利用</b>。例外包括：① <b>公務機關或學術研究機構</b>基於醫療、衛生或犯罪預防的目的，為統計或學術研究而有必要，且資料經提供者處理後，或依蒐集者的揭露方式，<b>無從識別特定當事人</b>；② <b>經當事人書面同意</b>（有但書限制）；另外還有法律明文規定、法定職務或義務等例外。</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>人體研究法 第 5 條</b> ＋ 衛福部公告</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">人體研究原則上，<b>研究前要先擬計畫，經倫理審查委員會（IRB）審查通過</b>；屬衛福部公告「得免審查」範圍的，才不在此限。</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>衛福部公告：得免倫理審查委員會審查之人體研究案件範圍</b>（101.7.5 衛署醫字第 1010265075 號）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">免審有<b>嚴格前提</b>：研究對象不能是未成年人、收容人、原住民、孕婦、身心障礙、精神病患或其他可能無法自由決定者；範圍包含「非記名、非互動、非介入性且<b>無從辨識特定個人</b>」的研究，以及風險極低、經 IRB 評估後核發免審證明者。<b>要不要免審，是由 IRB 判定，不是研究者自己決定。</b></td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>衛福部公告：得免取得研究對象同意之人體研究案件範圍</b>（101.7.5 衛署醫字第 1010265083 號）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">包含「自<b>合法生物資料庫</b>取得之<b>去連結或無法辨識特定個人</b>的資料、檔案、文件、資訊或檢體進行研究」，但<b>不包括涉及族群或群體利益者</b>。</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>衛福部函釋</b>（衛部醫字第 1030013183 號）</td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left">對於未成年人、收容人、原住民、孕婦、身心障礙、精神病患等族群的資料，<b>無論是否去連結</b>，都<b>不屬於</b>免審範圍；另外，針對檢體，「去連結」不等於可以轉作原始目的以外的用途。</td></tr><tr><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>「去連結」與「去識別化」</b></td><td style="border:1px solid #888;padding:6px 12px;background:#fff;color:#000;text-align:left"><b>去連結</b>：以匿名化等方式，讓資料<b>永久無法</b>以合理方式連結、比對出當事人；<b>去識別化</b>：資料經處理後，無從識別特定當事人。<b>只要還能連結回個人，就仍然是「可識別」</b>——把姓名換成代號、但自己手上保留對照表，通常還不算去連結。</td></tr></table>


**給未來研究者的三個實務提醒**
1. **先問你的 IRB**：這個研究是免審、簡易審查，還是一般審查？不要自己判斷。
2. **不要把含個資的資料貼到公開的 AI 服務**（包括聊天機器人），哪怕只是「幫我看一下這段病歷」。
3. 今天的資料集是**公開、已去識別**的；用**院內資料**做研究時，上面每一項都要處理。

資料來源：[個資法第 6 條（個人資料保護委員會籌備處）](https://www.pdpc.gov.tw/News_Content/100/293/)、[衛福部：得免倫理審查委員會審查之人體研究案件範圍](https://dep.mohw.gov.tw/DOMA/dl-16078-31bb0e95-2407-4323-9bbf-a62b7f8dcee3.html)、[臺北醫學大學轉知：衛福部函釋去連結檢體使用規範及免審範圍](https://ohr.tmu.edu.tw/index.php/zh-Hant/news/133)、[全國法規資料庫（人體研究法）](https://law.moj.gov.tw/)。

### 收尾：把今天帶走

**資料不只是數字，它還有來源與隱私**
- **去識別化**：影像臉部、DICOM 標頭、病歷文字（上一節）。研究前要經 IRB，並依個資法處理。
- **偏誤**：資料若只來自某一家醫院、某一族群，模型在別的地方可能就不準。跨院、跨儀器的外部驗證很重要。
- **標籤品質**：不同醫師對同一份 EEG 判讀不一致時，AI 學到的是「誰的標準」？

**閱讀 AI 論文時的檢查清單**
1. 資料從哪裡來？有多少醫院、多少病人？
2. 切分是以「病人」還是以「影像／切片」為單位？
3. 前處理（標準化、補值、挑特徵）是在切分**之前**還是**之後**做的？
4. 有沒有外部驗證（不同醫院、不同儀器）？
5. 標籤是誰標的？有幾位？一致性如何？
6. 評估指標適合這個疾病的盛行率嗎？（別只看 accuracy）

🎉 恭喜完成！你的 Google 雲端硬碟資料夾 `AI_preprocessing_class` 裡，有今天產生的所有檔案。